In [157]:
# Data import
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

df = pd.read_excel("pcos_data_base_ml.xlsx")


## PCOS Data Cleaning and Validation Pipeline – Step List

### Core Pipeline Steps

1. **Data overview** – shape, data types, missing values, group distribution
2. **Identification of indexable columns** – subject_id, group, etc.
3. **Parameters existing only in one group** – PCOS vs Control_no_PCOS
4. **Classification of parameters into categories** – hormonal, metabolic, lipid, inflammatory, etc.
5. **Parameters with very few data** – N < 10 (exclude) and 10 ≤ N < 30 (separate analysis)
6. **Formatting error detection** – commas instead of dots, text null values
7. **Parameters without variance** – constant or near-constant (unique values ≤ 2)
8. **Unit identification** – mg/dL vs mmol/L (glucose), ng/mL vs μg/L (D-dimer), nmol/L vs pmol/L (SHBG)
9. **Column consolidation** – merging parameters across groups and methods
10. **Outlier detection (biological plausibility)** – values outside physiological ranges (e.g., CRP > 100, SHBG > 300)
11. **Removal of highly incomplete data** – columns with >30% missing values
12. **Missing data imputation strategy** – mean/median, KNN, MICE, or deletion
13. **Normalization / scaling before clustering** – Z-score, Min-Max, or log transformation (for skewed distributions)

## Parameter Quality Dictionary – Initialization

A dedicated dictionary `parameter_quality` was created to track metadata and quality flags for every column in the dataset. This dictionary serves as the single source of truth for all data cleaning and preprocessing decisions.

### Dictionary Structure

Each column is represented by a nested dictionary containing the following keys:

| Key | Type | Description |
|-----|------|-------------|
| **Basic info** | | |
| `original_name` | str | Original column name from the dataset |
| `data_type` | str | Pandas data type (e.g., float64, int64, str) |
| `n_total` | int | Total number of rows in the dataset |
| `n_nonnull` | int | Number of non‑null values in the column |
| `n_unique` | int | Number of unique values in the column |
| **Descriptive statistics** | | |
| `mean` | float or None | Arithmetic mean (for numeric columns) |
| `median` | float or None | Median (for numeric columns) |
| `min` | float or None | Minimum value (for numeric columns) |
| `max` | float or None | Maximum value (for numeric columns) |
| `q1` | float or None | First quartile (25th percentile) |
| `q3` | float or None | Third quartile (75th percentile) |
| **Quality flags** | | |
| `useful` | bool | Overall flag indicating whether the column should be retained for analysis |
| `exclude_reason` | list | List of reasons why the column was excluded (if applicable) |
| **Specific issues** | | |
| `only_in_group` | str or None | `'PCOS'`, `'Control_no_PCOS'`, or `None` if present in both groups |
| `too_few_data` | bool | `True` if number of non‑null values is below threshold (e.g., < 30) |
| `no_variance` | bool | `True` if column has ≤ 2 unique values (constant or near‑constant) |
| `formatting_errors` | bool | `True` if column contains commas, text nulls, or other formatting issues |
| `unit_unknown` | bool | `True` if the measurement unit could not be determined |
| `outliers_detected` | bool | `True` if biologically implausible values were found |
| **Unit tracking** | | |
| `detected_unit` | str or None | Detected unit of measurement (e.g., `'mg/dL'`, `'nmol/L'`, `'ng/mL'`) |
| `original_unit` | str or None | Unit specified in metadata (if available) |
| **Merge tracking** | | |
| `merged_from` | list | List of source columns that were consolidated into this column |
| `merged_into` | str or None | Name of the consolidated column this column was merged into |
| **Decision metadata** | | |
| `category` | str or None | Clinical category assigned to the parameter (e.g., `'Thyroid'`, `'Lipid'`) |
| `recommended_action` | str or None | Suggested action: `'exclude'`, `'fix'`, `'merge'`, `'keep'` |
| `notes` | list | Free‑text notes documenting decisions and observations |

### Descriptive Statistics

For numeric columns (`float64`, `int64`), the following descriptive statistics are calculated at initialization and stored in the dictionary:

- **`mean`** – arithmetic mean (average)
- **`median`** – median value (50th percentile)
- **`min`** – minimum observed value
- **`max`** – maximum observed value
- **`q1`** – first quartile (25th percentile)
- **`q3`** – third quartile (75th percentile)

These statistics are used throughout the pipeline for:
- Unit detection (comparing median values to reference ranges)
- Outlier detection (identifying values outside physiological ranges)
- Data quality assessment (evaluating variance and distribution)
- Post‑consolidation validation (checking merged distributions)

In [158]:
# Initialize quality dictionary for all columns
parameter_quality = {}

for col in df.columns:
    # Calculate statistics for numeric columns
    if df[col].dtype in ['float64', 'int64']:
        series = df[col].dropna()
        mean_val = series.mean() if len(series) > 0 else None
        median_val = series.median() if len(series) > 0 else None
        min_val = series.min() if len(series) > 0 else None
        max_val = series.max() if len(series) > 0 else None
        q1 = series.quantile(0.25) if len(series) > 0 else None
        q3 = series.quantile(0.75) if len(series) > 0 else None
    else:
        mean_val = None
        median_val = None
        min_val = None
        max_val = None
        q1 = None
        q3 = None
    
    parameter_quality[col] = {
        # Basic info
        'original_name': col,
        'data_type': str(df[col].dtype),
        'n_total': len(df),
        'n_nonnull': df[col].notna().sum(),
        'n_unique': df[col].nunique(),
        
        # Descriptive statistics (for numeric columns)
        'mean': mean_val,
        'median': median_val,
        'min': min_val,
        'max': max_val,
        'q1': q1,
        'q3': q3,
        
        # Flags (start as True/empty – will be updated when problems found)
        'useful': True,           # Overall flag for further analysis
        'exclude_reason': [],     # List of reasons to exclude
        
        # Specific issues
        'only_in_group': None,    # 'PCOS', 'Control_no_PCOS', or None
        'too_few_data': False,    # N < 30
        'no_variance': False,     # ≤2 unique values
        'formatting_errors': False, # commas, text nulls
        'unit_unknown': False,    # cannot determine units
        'outliers_detected': False, # extreme values
        
        # Unit detection
        'detected_unit': None,    # e.g., 'mg/dL', 'nmol/L', 'ng/mL', etc.
        'original_unit': None,    # If specified in metadata
        
        # Merging tracking
        'merged_from': [],        # List of source columns that were merged into this column
        'merged_into': None,      # Name of the consolidated column this was merged into
        
        # Metadata for decision making
        'category': None,         # e.g., 'TSH', 'FT4'
        'recommended_action': None, # 'exclude', 'fix', 'merge', 'keep'
        'notes': []               # Free text notes
    }

print(f"Initialized quality dictionary for {len(parameter_quality)} parameters")

Initialized quality dictionary for 224 parameters


In [159]:
# PARAMETER QUALITY DICTIONARY – FIRST 10 COLUMNS
print("=" * 80)
print("PARAMETER QUALITY DICTIONARY – FIRST 10 COLUMNS")
print("=" * 80)

# Get first 10 columns
first_10_cols = list(df.columns)[:10]

# Create DataFrame for display
display_data = []
for col in first_10_cols:
    if col in parameter_quality:
        info = parameter_quality[col]
        display_data.append({
            'Parameter': col,
            'original_name': info.get('original_name'),
            'data_type': info.get('data_type'),
            'n_total': info.get('n_total'),
            'n_nonnull': info.get('n_nonnull'),
            'n_unique': info.get('n_unique'),
            'mean': info.get('mean'),
            'median': info.get('median'),
            'min': info.get('min'),
            'max': info.get('max'),
            'q1': info.get('q1'),
            'q3': info.get('q3'),
            'useful': info.get('useful'),
            'exclude_reason': info.get('exclude_reason'),
            'only_in_group': info.get('only_in_group'),
            'too_few_data': info.get('too_few_data'),
            'no_variance': info.get('no_variance'),
            'formatting_errors': info.get('formatting_errors'),
            'unit_unknown': info.get('unit_unknown'),
            'outliers_detected': info.get('outliers_detected'),
            'detected_unit': info.get('detected_unit'),
            'original_unit': info.get('original_unit'),
            'merged_from': info.get('merged_from'),
            'merged_into': info.get('merged_into'),
            'category': info.get('category'),
            'recommended_action': info.get('recommended_action'),
            'notes': info.get('notes')
        })

display_df = pd.DataFrame(display_data)

# Display all columns (no truncation)
pd.set_option('display.max_colwidth', 30)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

display(display_df)

# Save to CSV for full inspection
display_df.to_csv('parameter_quality_first_10.csv', index=False)
print("\n✅ Saved to 'parameter_quality_first_10.csv'")

PARAMETER QUALITY DICTIONARY – FIRST 10 COLUMNS


,Parameter,original_name,data_type,n_total,n_nonnull,n_unique,mean,median,min,max,q1,q3,useful,exclude_reason,only_in_group,too_few_data,no_variance,formatting_errors,unit_unknown,outliers_detected,detected_unit,original_unit,merged_from,merged_into,category,recommended_action,notes
0,Wiek,Wiek,int64,1331,1331,21,21.100676,21.000,16.00,41.000000,19.0000,23.000,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
1,group,group,str,1331,1331,2,NaN,NaN,NaN,NaN,NaN,NaN,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
2,subject_id,subject_id,str,1331,1331,1331,NaN,NaN,NaN,NaN,NaN,NaN,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
3,17 - OH progesteron (L79) ...,17 - OH progesteron (L79) ...,float64,1331,965,248,1.182603,0.990,0.20,8.783333,0.6900,1.430,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
4,17 OH progesteron (L79),17 OH progesteron (L79),float64,1331,212,172,2.069249,2.125,0.14,5.240000,1.3575,2.745,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
5,ALAT (ALT),ALAT (ALT),float64,1331,263,161,19.642395,16.400,5.00,127.000000,12.9500,22.000,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
6,AMH (hormon anty-Mullerows...,AMH (hormon anty-Mullerows...,float64,1331,976,585,6.406019,5.715,0.01,27.800000,3.7800,8.405,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
7,AMH ng/ml,AMH ng/ml,float64,1331,45,44,3.479333,3.040,1.12,9.430000,1.8900,4.760,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
8,AMH-anty Mullerian Hormon ...,AMH-anty Mullerian Hormon ...,float64,1331,147,139,6.959456,6.150,1.34,19.850000,4.3600,8.515,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]
9,APTT Czas kaolinowo-kefali...,APTT Czas kaolinowo-kefali...,float64,1331,5,5,34.540000,34.100,30.20,39.400000,33.8000,35.200,True,[],None,False,False,False,False,False,None,None,[],None,None,None,[]



✅ Saved to 'parameter_quality_first_10.csv'


* * *

## Step 1: Data Overview

In this step, a comprehensive initial assessment of the dataset was performed. The key characteristics examined included:

- **Dataset dimensions** – number of rows and columns
- **Data types** – distribution of numeric, categorical, and datetime columns
- **Group distribution** – proportion of PCOS patients versus healthy controls
- **Missing data** – total missing values, columns affected, and identification of highly incomplete columns (>80% missing)

This overview provided the foundation for all subsequent preprocessing decisions, including which columns to retain, how to handle missing data, and which parameters required special attention due to group‑specific availability.

In [160]:
# STEP 1: Data overview – shape, data types, missing values, group distribution

print("=" * 80)
print("STEP 1: DATA OVERVIEW – COMPREHENSIVE")
print("=" * 80)

# ============================================================================
# 1. BASIC INFORMATION
# ============================================================================
print("\n" + "=" * 80)
print("1. BASIC INFORMATION")
print("=" * 80)

print(f"Shape: {df.shape}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# 2. DATA TYPES
# ============================================================================
print("\n" + "=" * 80)
print("2. DATA TYPES")
print("=" * 80)

# Simplified dtype counts without frequency-specific types
dtype_counts = {}
for dtype in df.dtypes.unique():
    # Convert datetime64[us] to just datetime64 for counting
    dtype_name = str(dtype)
    if 'datetime64' in dtype_name:
        dtype_name = 'datetime64'
    dtype_counts[dtype_name] = dtype_counts.get(dtype_name, 0) + (df.dtypes == dtype).sum()

print("Data type counts:")
for dtype, count in sorted(dtype_counts.items(), key=lambda x: -x[1]):
    print(f"  {dtype}: {count}")

# Detailed breakdown
print("\nData type breakdown by category:")
for dtype, count in dtype_counts.items():
    # Get columns of this type (handle datetime64 specially)
    if 'datetime64' in dtype:
        cols = df.select_dtypes(include=['datetime64']).columns.tolist()
    else:
        cols = df.select_dtypes(include=[dtype]).columns.tolist()
    print(f"  {dtype}: {count} columns – {', '.join(cols[:5])}{'...' if len(cols) > 5 else ''}")

# ============================================================================
# 3. GROUP DISTRIBUTION
# ============================================================================
print("\n" + "=" * 80)
print("3. GROUP DISTRIBUTION")
print("=" * 80)

if 'group' in df.columns:
    group_counts = df['group'].value_counts()
    print(group_counts)
    print(f"\nPercentage:")
    for group, count in group_counts.items():
        print(f"  {group}: {count/len(df)*100:.1f}%")
else:
    print("No 'group' column found")

# ============================================================================
# 4. FIRST ROWS (PREVIEW)
# ============================================================================
print("\n" + "=" * 80)
print("4. FIRST 5 ROWS")
print("=" * 80)

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 30)
display(HTML(df.head().to_html()))

# ============================================================================
# 5. MISSING VALUES – COMPREHENSIVE
# ============================================================================
print("\n" + "=" * 80)
print("5. MISSING VALUES OVERVIEW")
print("=" * 80)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing': missing, 'Percent': missing_pct})
missing_summary = missing_summary[missing_summary['Missing'] > 0].sort_values('Missing', ascending=False)

print(f"Total missing values: {df.isnull().sum().sum():,}")
print(f"Columns with missing values: {len(missing_summary)}")
print(f"Columns with 0 missing values: {len(df.columns) - len(missing_summary)}")

# Columns with >80% missing (highly incomplete)
high_missing = missing_summary[missing_summary['Percent'] > 80]
print(f"\nColumns with >80% missing values: {len(high_missing)}")
if len(high_missing) > 0:
    for col, row in high_missing.head(20).iterrows():
        print(f"  • {col}: {row['Missing']} missing ({row['Percent']:.1f}%)")

# Top 20 columns with most missing values
print(f"\nTop 20 columns with most missing values:")
display(missing_summary.head(20))


STEP 1: DATA OVERVIEW – COMPREHENSIVE

1. BASIC INFORMATION
Shape: (1331, 224)
Rows: 1331
Columns: 224
Memory usage: 3.34 MB

2. DATA TYPES
Data type counts:
  float64: 191
  str: 30
  datetime64: 2
  int64: 1

Data type breakdown by category:
  int64: 1 columns – Wiek
  str: 30 columns – group, subject_id, Anty - HCV (L_ANTHCV), Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (COV2_G), Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (GRYA)...
  float64: 191 columns – 17 - OH progesteron (L79) (17-OHPG), 17 OH progesteron (L79), ALAT (ALT), AMH (hormon anty-Mullerowski) (AMH_CP), AMH ng/ml...
  datetime64: 2 columns – Przyjęcie na oddział zlecający, Wypis z oddziału zlecającego

3. GROUP DISTRIBUTION
group
PCOS               1286
Control_no_PCOS      45
Name: count, dtype: int64

Percentage:
  PCOS: 96.6%
  Control_no_PCOS: 3.4%

4. FIRST 5 ROWS


,Wiek,group,subject_id,17 - OH progesteron (L79) (17-OHPG),17 OH progesteron (L79),ALAT (ALT),AMH (hormon anty-Mullerowski) (AMH_CP),AMH ng/ml,AMH-anty Mullerian Hormon (AMH),APTT Czas kaolinowo-kefalinowy (APTTCZ),ASO - ilościowo (ASOIL),ASPAT (AST),Androstendion (ANDRO),Androstendion (I31),Androstendion (I31) (ANDRO),Anty - HCV (L_ANTHCV),Anty-TG (O18),Anty-TG (p/c przeciw tyreoglobulinie) (ATG),Anty-TPO (ATA_TPO),BMI kg/m2,Białko C-reaktywne (CRP),Bilirubina całkowita (TBIL),CEA (CEA),Ca125 (CA125),Ca19.9 (CA199),Cholesterol całkowity (TCHOL),D-dimery (DDIMER),DHEAS (DHEA),DHEAS ug/dl,Dobowy rytm tolerancji glukozy (L_G1030),Dobowy rytm tolerancji glukozy (L_G1200),Dobowy rytm tolerancji glukozy (L_G1500),Dobowy rytm tolerancji glukozy (L_G1800),Dobowy rytm tolerancji glukozy (L_G2100),Dobowy rytm tolerancji glukozy (L_G2400),Dobowy rytm tolerancji glukozy (L_G330),Dobowy rytm tolerancji glukozy (L_G700),Dokument (NAZWA),Estradiol (ESTRA),Estradiol (L_ESTRA),FSH (FSH),FSH 0' 30' 60' (L_FSH0),FSH 0' 30' 60' (L_FSH30),FSH 0' 30' 60' (L_FSH60),FSH IU/l,FT3 (FT3),FT4 (FT4),Ferrytyna (FERR),Ferrytyna (L05),Fibrynogen (L_FIB),Fosforany nieorganiczne (FOSFOR),GH (hormon wzrostu) (L_GH),Gamma Glutamylotranspeptydaza (GGTP),Glukoza (L_GLU),"Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (COV2_G)","Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (GRYA)","Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (GRYB)","Grypa A, B, RSV, SARS-CoV2 antygen - badanie jakościowe (GRYRSV)",HBSAG (L_HBSAG),HBsAg (L_HBSAG),HDL cholesterol (HDL),HbsAg (HBSAG),Hemoglobina glikowana (HBA1C_1),Hemoglobina glikowana (HBA1C_2),Hemoglobina glikowana (L53.IFC),Hemoglobina glikowana (L55),IGF-1 (insulinopodobny czynnik wzrostu 1) (IGF1_L),Insulina (INSUL),Insulina po 75g glukozy (3pkt.) (INSUL_0),Insulina po 75g glukozy (3pkt.) (INSUL_1),Insulina po 75g glukozy (3pkt.) (INSUL_2),"Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)","Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)","Jonogram (Sód, potas, chlorki) (L_CI)","Jonogram (Sód, potas, chlorki) (L_K)","Jonogram (Sód, potas, chlorki) (L_NA)",Kortyzol godz. 08:00 (KOR),Kortyzol godz. 23:00 (KOR23),Kortyzol na czczo (KORCZ),Kortyzol po Dexamethasonie (KORD),Kortyzol po Dexamethasonie [1mg] (KORD),Kortyzol po Dexamethasonie [2mg] (KOR2DXM),Kortyzol po Dexamethasonie [4mg] (KOR4DXM),Kreatynina (KREAT),Krzywa cukrowa - 2 punktowa (GLU120),Krzywa cukrowa - 2 punktowa (L_GLU_0),LDL cholesterol (LDL),LH (LH),LH 0' 30' 60' (LH0),LH 0' 30' 60' (L_LH30),LH 0' 30' 60' (L_LH60),LH IU/l,Limf 10^3/uL,MDRD GFR (MDRD),MPV fL,Mocz - badanie ogólne (MBARWA),Mocz - badanie ogólne (MBIALK),Mocz - badanie ogólne (MBILI),Mocz - badanie ogólne (MCIEZA),Mocz - badanie ogólne (MCUKIE),Mocz - badanie ogólne (MERY),Mocz - badanie ogólne (MKETON),Mocz - badanie ogólne (MLEU),Mocz - badanie ogólne (MNIT),Mocz - badanie ogólne (MOSAD),Mocz - badanie ogólne (MPH),Mocz - badanie ogólne (MPRZEJ),Mocz - badanie ogólne (MUROBI),Morfologia CBC (HCT),Morfologia CBC (HGB),Morfologia CBC (MCH),Morfologia CBC (MCHC),Morfologia CBC (MCV),Morfologia CBC (MPV),Morfologia CBC (NRBC_B),Morfologia CBC (NRBC_P),Morfologia CBC (PCT),Morfologia CBC (PDW),Morfologia CBC (PLCR),Morfologia CBC (PLT),Morfologia CBC (RBC),Morfologia CBC (RDW),Morfologia CBC (RDWSD),Morfologia CBC (WBC),"Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASO)","Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASOB)","Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (EOS#)","Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (EOS


5. MISSING VALUES OVERVIEW
Total missing values: 230,021
Columns with missing values: 221
Columns with 0 missing values: 3

Columns with >80% missing values: 162
  • TK jamy brzusznej i miednicy z kontrastem (nan): 1331.0 missing (100.0%)
  • TK jamy brzusznej z kontrastem (nan): 1331.0 missing (100.0%)
  • Mocz - badanie ogólne (MOSAD): 1331.0 missing (100.0%)
  • RTG kości nadgarstka, PA, bok (nan): 1331.0 missing (100.0%)
  • USG piersi (nan): 1331.0 missing (100.0%)
  • NLR: 1331.0 missing (100.0%)
  • USG tarczycy i przytarczyc (nan): 1331.0 missing (100.0%)
  • USG wątroby – elastografia (nan): 1331.0 missing (100.0%)
  • USG nadnerczy (nan): 1331.0 missing (100.0%)
  • USG brzucha i przestrzeni zaotrzewnowej (nan): 1331.0 missing (100.0%)
  • Wymaz z kanału szyjki macicy - posiew mykologiczny (Wynik badania): 1331.0 missing (100.0%)
  • PLR: 1331.0 missing (100.0%)
  • Wymaz z kanału szyjki macicy - posiew tlenowy (Wynik badania): 1331.0 missing (100.0%)
  • Grypa A, B, RSV, SA

,Missing,Percent
TK jamy brzusznej i miednicy z kontrastem (nan),1331,100.000000
TK jamy brzusznej z kontrastem (nan),1331,100.000000
Mocz - badanie ogólne (MOSAD),1331,100.000000
"RTG kości nadgarstka, PA, bok (nan)",1331,100.000000
USG piersi (nan),1331,100.000000
NLR,1331,100.000000
USG tarczycy i przytarczyc (nan),1331,100.000000
USG wątroby – elastografia (nan),1331,100.000000
USG nadnerczy (nan),1331,100.000000
USG brzucha i przestrzeni zaotrzewnowej (nan),1331,100.000000


* * *

## Step 2: Identification of Indexable Columns

This step separates columns that serve as identifiers or demographic variables from clinical parameters.

**Indexable columns** – identifiers and grouping variables:
- `group` – PCOS / Control_no_PCOS
- `subject_id` – unique patient identifier
- `Nr KG` – hospital case number

These columns are excluded from preprocessing steps such as normalization or imputation and are treated separately in downstream analyses.

In [161]:
# STEP 2: IDENTIFICATION OF INDEXABLE COLUMNS


print("\n" + "=" * 80)
print("STEP 2: IDENTIFICATION OF INDEXABLE COLUMNS")
print("=" * 80)

indexable_cols = ['group', 'subject_id', 'Nr KG']

# Process indexable columns
found_indexable = []
for col in indexable_cols:
    if col in df.columns:
        found_indexable.append(col)
        print(f"✓ {col} (indexable)")
    else:
        print(f"✗ {col} (not found)")

# Mark indexable columns in parameter_quality
for col in found_indexable:
    if col in parameter_quality:
        parameter_quality[col]['category'] = 'Index'
        parameter_quality[col]['recommended_action'] = 'keep_as_index'
        parameter_quality[col]['notes'].append('Used as identifier/grouping variable')
        
print(f"✅ Identified {len(found_indexable)} indexable columns: {found_indexable}\n")
        


STEP 2: IDENTIFICATION OF INDEXABLE COLUMNS
✓ group (indexable)
✓ subject_id (indexable)
✓ Nr KG (indexable)
✅ Identified 3 indexable columns: ['group', 'subject_id', 'Nr KG']



* * *

## Step 3: Parameters Existing Only in One Group

This step identifies which clinical parameters are available only in PCOS patients, only in controls, or in both groups. This information is critical for downstream analyses, as parameters restricted to a single group cannot be used for cross‑group comparisons.

### Results Summary

| Category | Count | Description |
|----------|-------|-------------|
| **Only in PCOS** | 190 | Parameters measured exclusively in PCOS patients |
| **Only in Control_no_PCOS** | 18 | Parameters measured exclusively in healthy controls |
| **In both groups** | 3 | Demographic/identifier columns (Wiek, group, subject_id) |
| **In neither group** | 13 | Completely empty columns (no data in either group) |

### Only in PCOS (190 parameters)

These include hormonal, metabolic, and hematological parameters measured as part of PCOS diagnostic workup. Examples:
- AMH, androstenedione, anti-TPO, SHBG (L_SHGB), testosterone, insulin, cortisol, etc.

### Only in Control_no_PCOS (18 parameters)

These are typically measured only in the control group and represent the same clinical parameters but using different laboratory codes or measurement protocols. Examples:
- `AMH ng/ml`, `DHEAS ug/dl`, `FSH IU/l`, `LH IU/l`, `testosteron ng/ml`, `glu 0' mg/dl`, `insulina uU/ml`

### In Both Groups (3 parameters)

Only demographic and identifier columns contain data in both groups:
- `Wiek` (age)
- `group` (diagnostic group assignment)
- `subject_id` (patient identifier)

### In Neither Group (13 parameters)

These columns are completely empty (100% missing) and will be excluded from further analysis. They consist mainly of:
- Imaging studies (USG, TK, RTG)
- Microbiology swabs
- Calculated ratios (NLR, PLR)

### Implications

| Parameter Type | Recommendation |
|----------------|----------------|
| Only in PCOS | Can be used for PCOS‑specific analyses, but not for PCOS vs Control comparisons |
| Only in Control | Can be used for Control‑specific analyses, but not for cross‑group comparisons |
| In both groups | Suitable for cross‑group comparisons (only demographic variables) |
| In neither group | Exclude entirely from analysis |

In [162]:
# STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP

print("\n" + "=" * 80)
print("STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP")
print("=" * 80)

pcos_df = df[df['group'] == 'PCOS']
control_df = df[df['group'] == 'Control_no_PCOS']

only_pcos = []
only_control = []
in_both = []
in_neither = []

for col in df.columns:
    pcos_count = pcos_df[col].notna().sum()
    control_count = control_df[col].notna().sum()
    
    if pcos_count > 0 and control_count == 0:
        only_pcos.append((col, pcos_count))
        parameter_quality[col]['only_in_group'] = 'PCOS'
        parameter_quality[col]['notes'].append(f'Has {pcos_count} values, but 0 in Control')
        
    elif control_count > 0 and pcos_count == 0:
        only_control.append((col, control_count))
        parameter_quality[col]['only_in_group'] = 'Control_no_PCOS'
        parameter_quality[col]['notes'].append(f'Has {control_count} values, but 0 in PCOS')
        
    elif pcos_count > 0 and control_count > 0:
        in_both.append((col, pcos_count, control_count))
        parameter_quality[col]['only_in_group'] = None  # both groups
        parameter_quality[col]['notes'].append(f'Has {pcos_count} values in PCOS and {control_count} in Control')
        
    else:  # pcos_count == 0 and control_count == 0
        in_neither.append(col)
        parameter_quality[col]['only_in_group'] = None
        parameter_quality[col]['notes'].append('No values in either group')

# Display results
print(f"\n📊 ONLY IN PCOS ({len(only_pcos)} parameters):")
for col, count in only_pcos[:20]:
    print(f"  • {col}: {count} values")
if len(only_pcos) > 20:
    print(f"  ... and {len(only_pcos) - 20} more")

print(f"\n📊 ONLY IN Control_no_PCOS ({len(only_control)} parameters):")
for col, count in only_control[:20]:
    print(f"  • {col}: {count} values")
if len(only_control) > 20:
    print(f"  ... and {len(only_control) - 20} more")

print(f"\n📊 IN BOTH GROUPS ({len(in_both)} parameters):")
for col, pcos_n, control_n in in_both[:20]:
    print(f"  • {col}: PCOS={pcos_n}, Control={control_n}")
if len(in_both) > 20:
    print(f"  ... and {len(in_both) - 20} more")

print(f"\n📊 IN NEITHER GROUP ({len(in_neither)} parameters):")
for col in in_neither[:20]:
    print(f"  • {col}")
if len(in_neither) > 20:
    print(f"  ... and {len(in_neither) - 20} more")


STEP 3: PARAMETERS EXISTING ONLY IN ONE GROUP

📊 ONLY IN PCOS (190 parameters):
  • 17 - OH progesteron (L79) (17-OHPG): 965 values
  • 17 OH progesteron (L79): 212 values
  • ALAT (ALT): 263 values
  • AMH (hormon anty-Mullerowski) (AMH_CP): 976 values
  • AMH-anty Mullerian Hormon (AMH): 147 values
  • APTT Czas kaolinowo-kefalinowy (APTTCZ): 5 values
  • ASO - ilościowo (ASOIL): 1 values
  • ASPAT (AST): 263 values
  • Androstendion (ANDRO): 682 values
  • Androstendion (I31): 189 values
  • Androstendion (I31) (ANDRO): 255 values
  • Anty - HCV (L_ANTHCV): 1 values
  • Anty-TG (O18): 21 values
  • Anty-TG (p/c przeciw tyreoglobulinie) (ATG): 254 values
  • Anty-TPO (ATA_TPO): 1043 values
  • Białko C-reaktywne (CRP): 19 values
  • Bilirubina całkowita (TBIL): 76 values
  • CEA (CEA): 1 values
  • Ca125 (CA125): 41 values
  • Ca19.9 (CA199): 1 values
  ... and 170 more

📊 ONLY IN Control_no_PCOS (18 parameters):
  • AMH ng/ml: 45 values
  • BMI kg/m2: 45 values
  • DHEAS ug/dl: 45 

***

## Step 4: Classification of Parameters into Clinical Categories

This step assigns each clinical parameter to a predefined category based on its physiological function and clinical relevance. Categories include hormonal (e.g., testosterone, AMH, prolactin), metabolic (e.g., glucose, insulin), lipid, inflammatory, thyroid, adrenal, hematological, and others.

### Method

Classification is performed using a two-stage approach:

1. **Keyword matching** – each category has a list of keywords (e.g., `testosteron`, `insulina`, `crp`). If a keyword appears in the column name, the column is assigned to that category.
2. **Regex pattern matching** – if no keyword matches, regular expressions are used to capture variations (e.g., `l_testos` for testosterone, `l_g[0-9]+` for OGTT glucose).

### Categories Defined

| Category Group | Examples |
|----------------|----------|
| **Androgen / Hormonal** | Testosterone, SHBG, DHEAS, Androstenedione, AMH, LH, FSH, Estradiol, Progesterone_17OHP, Prolactin |
| **Metabolic / Glucose** | Glucose_Fasting, Glucose_OGTT, Insulin_Fasting, Insulin_OGTT, HbA1c, TyG |
| **Lipid** | Cholesterol_Total, HDL, LDL, Triglycerides |
| **Thyroid** | TSH, FT3, FT4, Anti_TPO, Anti_TG, TRAb |
| **Adrenal / Cortisol** | Cortisol_8AM, Cortisol_11PM, Cortisol_Dexamethasone, ACTH_Stimulation |
| **Inflammatory / Hematology** | CRP, NLR, PLR, Neutrophils, Lymphocytes, Monocytes, Eosinophils, Basophils, Platelets, WBC, RBC, Hemoglobin, Hematocrit, MCV, MCH, MCHC, RDW, MPV, Ferritin, Iron |
| **Liver** | ALT, AST, GGTP, Bilirubin |
| **Kidney** | Creatinine, GFR_MDRD, Electrolytes |
| **Coagulation** | D_Dimer, Fibrinogen, PT_APTT |
| **Growth Factors** | IGF1, GH |
| **Tumor Markers** | CA125, CA19_9, CEA, HE4_ROMA |
| **Immunology** | Anti_HCV, HBsAg, Anti_GAD, ASO |
| **Urinalysis** | Urinalysis parameters |
| **Imaging** | Ultrasound, CT_Scan, XRay |
| **Microbiology** | Microbiology cultures and swabs |
| **Vitamins** | Vitamin_D, Calcium, Phosphate |
| **Other** | Beta_HCG, Virology, COVID_Respiratory, Immature_Granulocytes |

### Results

After automatic classification:

- **All 224 columns** were assigned to a category
- **No uncategorized columns** remained
- Category distribution was reviewed and documented

### Manual Corrections Applied

Despite accurate automatic classification, several manual corrections were applied to improve clinical coherence:

| Correction | Original Category | New Category | Reason |
|------------|-------------------|--------------|--------|
| IGF-1 → IGF1 | Insulin_Fasting | IGF1 | IGF-1 is a growth factor, not insulin |
| TK → CT_Scan | AST | CT_Scan | Imaging study, not liver enzyme |
| USG elastografia → Ultrasound | AST | Ultrasound | Imaging study, not liver enzyme |
| TRAb → Thyroid_Autoantibodies | TSH | Thyroid_Autoantibodies | Autoantibody, not thyroid hormone |
| MCHC → MCHC | MCH | MCHC | Separate red cell index |
| NRBC → NRBC | RBC | NRBC | Nucleated RBCs are distinct from mature RBCs |

### Final Verification

After manual corrections, the updated categorization list was reviewed and confirmed. All columns are now correctly assigned to their respective clinical categories, ready for downstream preprocessing steps.

In [ ]:
# STEP 4: CLASSIFICATION OF PARAMETERS INTO CLINICAL CATEGORIES

import re

print("=" * 80)
print("STEP 4: CLASSIFY PARAMETERS INTO CLINICAL CATEGORIES")
print("=" * 80)

# Define specific categories with keywords and patterns
category_mapping = {
    # Index / Demographics
    'Other': {
        'keywords': ['wiek', 'BMI kg/m2', 'rok kg', 'dokument', 'przyjęcie', 'wypis', 'oddział'],
        'patterns': []
    },
    
    'Virology': {
        'keywords': ['grypa', 'covid', 'sars-cov', 'rsv', 'cov2'],
        'patterns': [r'grypa', r'covid', r'sars-cov', r'rsv', r'cov2']
    },
    'XRay': {
        'keywords': ['rtg', 'x-ray', 'radiograph'],
        'patterns': [r'rtg', r'x-ray']
    },
    'GGT': {
        'keywords': ['ggtp', 'gamma gt', 'gamma-glutamyl'],
        'patterns': [r'ggtp', r'gamma']
    },
    
    # Androgen / Hormonal
    'Testosterone': {
        'keywords': ['testosteron', 'testosterone'],
        'patterns': [r'l_testos', r'\btest(-| )?free\b']
    },
    'Testosterone_Free': {
        'keywords': ['wolny testosteron', 'free testosterone', 'test-f', 'o41.w'],
        'patterns': [r'wolny testosteron', r'free testosterone', r'test-f', r'o41\.w']
    },
    'SHBG': {
        'keywords': ['shbg', 'sex hormone binding globulin'],
        'patterns': [r'shbg', r'l_shgb']
    },
    'DHEAS': {
        'keywords': ['dheas', 'dhea', 'dehydroepiandrosterone'],
        'patterns': [r'dheas', r'dhea']
    },
    'Androstenedione': {
        'keywords': ['androstendion', 'androstenedione'],
        'patterns': [r'\bandrosten(dion|dione)\b', r'\bandro\b']
    },
    'AMH': {
        'keywords': ['amh', 'anty-mullerian', 'anti-mullerian'],
        'patterns': [r'amh', r'anty-mullerian', r'anti-mullerian']
    },
    'LH': {
        'keywords': ['lh', 'luteinizing hormone'],
        'patterns': [r'lh', r'l_lh']
    },
    'FSH': {
        'keywords': ['fsh', 'follicle stimulating'],
        'patterns': [r'fsh', r'l_fsh']
    },
    'Estradiol': {
        'keywords': ['estradiol', 'e2', 'estra'],
        'patterns': [r'estradiol', r'estra', r'e2']
    },
    'Progesterone_17OHP': {
        'keywords': ['17-oh progesteron', '17 ohp', 'l79', 'synacthen', '17-ohp'],
        'patterns': [r'17-oh', r'17ohp', r'l79', r'synacthen']
    },
    'Prolactin': {
        'keywords': ['prl', 'prolactin', 'makroprl'],
        'patterns': [r'prl', r'prolactin', r'makroprl']
    },
    
    # Metabolic / Glucose
    'Glucose_Fasting': {
        'keywords': ['glu 0', 'glucose 0', 'glukoza', 'l_glu_0', 'glu 0\''],
        'patterns': [r'glu 0', r'glucose 0', r'l_glu_0', r'glu 0\'', r'glukoza']
    },
    'Glucose_OGTT': {
        'keywords': ['dobowy rytm tolerancji glukozy', 'krzywa cukrowa'],
        'patterns': [r'l_g[0-9]+', r'ogtt']
    },
    'Insulin_Fasting': {
        'keywords': ['insulina', 'insulin', 'insul', 'insulina uu/ml'],
        'patterns': [r'insulina', r'insulin', r'insul', r'insulina uu']
    },
    'Insulin_OGTT': {
        'keywords': ['insulina po', 'l97', 'insulin po 75g'],
        'patterns': [r'l97', r'insulina po', r'insulin po 75g']
    },
    'HOMA_IR': {
        'keywords': ['homa', 'homa-ir'],
        'patterns': [r'homa']
    },
    'QUICKI': {
        'keywords': ['quicki'],
        'patterns': [r'quicki']
    },
    'HbA1c': {
        'keywords': ['hemoglobina glikowana', 'hba1c', 'l53.ifc', 'glycated hemoglobin'],
        'patterns': [r'hba1c', r'hemoglobina glikowana', r'l53', r'glycated']
    },
    'TyG': {
        'keywords': ['tyg', 'triglyceride glucose'],
        'patterns': [r'tyg']
    },
    
    # Thyroid
    'TSH': {
        'keywords': ['tsh', 'thyrotropin'],
        'patterns': [r'tsh']
    },
    'FT3': {
        'keywords': ['ft3', 'free t3'],
        'patterns': [r'ft3']
    },
    'FT4': {
        'keywords': ['ft4', 'free t4'],
        'patterns': [r'ft4']
    },
    'Anti_TPO': {
        'keywords': ['anty-tpo', 'anti-tpo', 'ata_tpo'],
        'patterns': [r'anty-tpo', r'anti-tpo', r'tpo']
    },
    'Anti_TG': {
        'keywords': ['anty-tg', 'anti-tg', 'antytyreoglobulinowe', 'atg'],
        'patterns': [r'anty-tg', r'anti-tg', r'antytyreoglobulinowe', r'\batg\b']
    },
    'TRAb': {
        'keywords': ['trab', 'tsh receptor antibody', 'przeciw receptorowi tsh'],
        'patterns': [r'trab', r'tsh receptor']
    },
    
    # Lipid
    'Cholesterol_Total': {
        'keywords': ['cholesterol całkowity', 'total cholesterol', 'tchol'],
        'patterns': [r'cholesterol całkowity', r'total cholesterol', r'tchol']
    },
    'HDL': {
        'keywords': ['hdl', 'hdl cholesterol'],
        'patterns': [r'hdl']
    },
    'LDL': {
        'keywords': ['ldl', 'ldl cholesterol'],
        'patterns': [r'ldl']
    },
    'Triglycerides': {
        'keywords': ['triglicerydy', 'triglycerides'],
        'patterns': [r'triglicerydy', r'triglycerides', r'\btg\b']
    },

    # Growth Factors
    'IGF1': {
        'keywords': ['igf-1', 'igf1', 'insulin like growth factor'],
        'patterns': [r'igf-?1', r'insulin like growth']
    },
    'GH': {
        'keywords': ['gh', 'growth hormone', 'hormon wzrostu'],
        'patterns': [r'\bgh\b', r'growth hormone', r'hormon wzrostu']  # \bgh\b zamiast gh
    },
    
    # Inflammatory / Hematology
    'CRP': {
        'keywords': ['crp', 'białko c-reaktywne', 'c-reactive protein'],
        'patterns': [r'crp', r'białko c-reaktywne']
    },
    'NLR': {
        'keywords': ['nlr', 'neutrophil lymphocyte ratio'],
        'patterns': [r'nlr']
    },
    'PLR': {
        'keywords': ['plr', 'platelet lymphocyte ratio'],
        'patterns': [r'plr']
    },
    'Neutrophils': {
        'keywords': ['neut', 'neutrofile', 'neutrophil'],
        'patterns': [r'neu', r'neut', r'neutrofil']
    },
    'Lymphocytes': {
        'keywords': ['limf', 'lim#', 'lymph', 'lymphocyte'],
        'patterns': [r'\blim#\b', r'\blim\b', r'limf', r'lymph']
    },
    'Monocytes': {
        'keywords': ['mono', 'monocyte'],
        'patterns': [r'mon', r'monocyte']
    },
    'Eosinophils': {
        'keywords': ['eos', 'eosinophil'],
        'patterns': [r'eos']
    },
    'Basophils': {
        'keywords': ['baso', 'basophil'],
        'patterns': [r'baso']
    },
    'Platelets': {
        'keywords': ['plt', 'płytki', 'thrombocyte', 'plt 10^3'],
        'patterns': [r'plt', r'płytki']
    },
    'WBC': {
        'keywords': ['wbc', 'leukocyty', 'white blood cell'],
        'patterns': [r'wbc', r'leukocyty']
    },
    'RBC': {
        'keywords': ['rbc', 'erythrocyte', 'krwinki czerwone'],
        'patterns': [r'rbc']
    },
    'Hemoglobin': {
        'keywords': ['hgb', 'hemoglobina'],
        'patterns': [r'hgb', r'hemoglobina']
    },
    'Hematocrit': {
        'keywords': ['hct', 'hematokryt'],
        'patterns': [r'hct', r'hematokryt']
    },
    'MCV': {
        'keywords': ['mcv', 'mean corpuscular volume'],
        'patterns': [r'mcv']
    },
    'MCH': {
        'keywords': ['mch', 'mean corpuscular hemoglobin'],
        'patterns': [r'mch']
    },
    'MCHC': {
        'keywords': ['mchc', 'mean corpuscular hemoglobin concentration'],
        'patterns': [r'mchc']
    },
    'RDW': {
        'keywords': ['rdw', 'red cell distribution width'],
        'patterns': [r'rdw']
    },
    'MPV': {
        'keywords': ['mpv', 'mean platelet volume', 'pct', 'pdw', 'plcr'],
        'patterns': [r'\bmpv\b', r'\bpct\b', r'\bpdw\b', r'\bplcr\b']
    },
    'Ferritin': {
        'keywords': ['ferrytyna', 'ferritin', 'ferr', 'l05'],
        'patterns': [r'ferrytyna', r'ferritin', r'ferr', r'l05']
    },
    'Iron': {
        'keywords': ['żelazo', 'iron', 'fe'],
        'patterns': [r'żelazo', r'iron', r'fe\b']
    },
    
    # Coagulation
    'D_Dimer': {
        'keywords': ['d-dimer', 'ddimer'],
        'patterns': [r'd-dimer', r'ddimer']
    },
    'Fibrinogen': {
        'keywords': ['fibrynogen', 'fibrinogen', 'l_fib'],
        'patterns': [r'fibrynogen', r'fibrinogen', r'l_fib']
    },
    'PT_APTT': {
        'keywords': ['protrombina', 'pt', 'aptt', 'inr'],
        'patterns': [r'protrombina', r'\bpt\b', r'aptt', r'inr']
    },
    
    # Liver
    'ALT': {
        'keywords': ['alt', 'alat'],
        'patterns': [r'alt', r'alat']
    },
    'AST': {
        'keywords': ['ast', 'aspat'],
        'patterns': [r'ast', r'aspat']
    },
    'GGTP': {
        'keywords': ['ggtp', 'gamma gt'],
        'patterns': [r'ggtp', r'gamma']
    },
    'Bilirubin': {
        'keywords': ['bilirubina', 'bilirubin'],
        'patterns': [r'bilirubina', r'bilirubin']
    },
    
    # Kidney
    'Creatinine': {
        'keywords': ['kreatynina', 'creatinine', 'kreat'],
        'patterns': [r'kreatynina', r'creatinine', r'kreat']
    },
    'GFR_MDRD': {
        'keywords': ['mdrd', 'gfr'],
        'patterns': [r'mdrd', r'gfr']
    },
    'Electrolytes': {
        'keywords': ['sód', 'potas', 'chlorki', 'jonogram'],
        'patterns': [r'sód', r'potas', r'chlorki', r'jonogram', r'\bl_na\b', r'\bl_k\b', r'\bl_cl\b']
    },
    
    
    # Adrenal / Cortisol
    'Cortisol_8AM': {
        'keywords': ['kortyzol 8', 'kortyzol godz 08', 'kor 8', 'kortyzol 8.00', 'korcz'],
        'patterns': [r'kortyzol.*8', r'kor.*8', r'kor08', r'korcz']
    },
    'Cortisol_11PM': {
        'keywords': ['kortyzol 23', 'kortyzol godz 23', 'kor23'],
        'patterns': [r'kortyzol.*23', r'kor23']
    },
    'Cortisol_Dexamethasone': {
        'keywords': ['kord', 'dexamethasone', 'deksametazon', 'kortyzol po dex'],
        'patterns': [r'kord', r'dexamethasone', r'deksametazon', r'kortyzol po']
    },
    'ACTH_Stimulation': {
        'keywords': ['synacthen', 'l79', 'test z synacthenem'],
        'patterns': [r'synacthen', r'l79', r'test z synacthenem']
    },
    
    # Tumor Markers
    'CA125': {
        'keywords': ['ca125', 'ca 125'],
        'patterns': [r'ca125', r'ca 125']
    },
    'CA19_9': {
        'keywords': ['ca19.9', 'ca 19.9'],
        'patterns': [r'ca19\.9', r'ca 19\.9']
    },
    'CEA': {
        'keywords': ['cea'],
        'patterns': [r'cea']
    },
    'HE4_ROMA': {
        'keywords': ['he4', 'roma', 'test roma'],
        'patterns': [r'he4', r'roma']
    },
    
    # Immunology / Autoimmunity
    'Anti_HCV': {
        'keywords': ['anty-hcv', 'anti-hcv', 'l_anthcv'],
        'patterns': [r'anty-hcv', r'anti-hcv', r'anthcv']
    },
    'HBsAg': {
        'keywords': ['hbsag', 'hbs ag', 'hepatitis b'],
        'patterns': [r'hbsag', r'hbs']
    },
    'Anti_GAD': {
        'keywords': ['pcgad', 'anti-gad', 'przeciw gad'],
        'patterns': [r'pcgad', r'anti-gad', r'przeciw gad']
    },
    'ASO': {
        'keywords': ['aso', 'antistreptolysin'],
        'patterns': [r'aso']
    },
    
    # Urinalysis
    'Urinalysis': {
        'keywords': ['mocz', 'urine'],
        'patterns': [r'\bmocz\b', r'\burine\b', r'\bmb\w+\b', r'\bmcieza\b', r'\bmcukie\b', r'\bmosad\b', r'\bmph\b', r'\bmleu\b', r'\bmnit\b', r'\bmery\b', r'\bmketon\b', r'\bmbarwa\b', r'\bmbialk\b', r'\bmprzej\b', r'\bmurobi\b']
    },
    
    # Imaging
    'Ultrasound': {
        'keywords': ['usg', 'ultrasound', 'elastografia'],
        'patterns': [r'usg', r'ultrasound', r'elastografia']
    },
    'CT_Scan': {
        'keywords': ['tk jamy', 'ct scan', 'tomografia', 'tk'],
        'patterns': [r'tk jamy', r'tomografia', r'\btk\b']
    },
    
    # Microbiology
    'Microbiology': {
        'keywords': ['posiew', 'wymaz', 'culture', 'swab', 'dat-zak', 'uwagi'],
        'patterns': [r'posiew', r'wymaz', r'culture', r'swab', r'dat-zak', r'uwagi']
    },
    
    # Vitamins / Other
    'Vitamin_D': {
        'keywords': ['witamina d', 'vitamin d', '25-hydroksywitamina', 'witd'],
        'patterns': [r'witamina d', r'vitamin d', r'25-hydroksy', r'witd']
    },
    'Calcium': {
        'keywords': ['wapń', 'calcium', 'tcal', 'l_cazjs'],
        'patterns': [r'wapń', r'calcium', r'tcal', r'cazjs']
    },
    'Phosphate': {
        'keywords': ['fosforany', 'phosphate', 'fosfor'],
        'patterns': [r'fosforany', r'phosphate', r'fosfor']
    },
    'Beta_HCG': {
        'keywords': ['beta hcg', 'bhcg', 'l_hcg'],
        'patterns': [r'beta hcg', r'bhcg', r'l_hcg']
    },
    
    # COVID / Respiratory
    'COVID_Respiratory': {
        'keywords': ['covid', 'sars-cov', 'grypa', 'rsv', 'cov2'],
        'patterns': [r'covid', r'sars-cov', r'grypa', r'rsv', r'cov2']
    },
    
    'Immature_Granulocytes': {
        'keywords': ['ig_b', 'ig_p'],
        'patterns': [r'\big_b\b', r'\big_p\b']
    },
}

# Initialize category assignment
for col, info in parameter_quality.items():
    if col in ['group', 'subject_id', 'Nr KG']:
        info['category'] = 'Index'
        continue
    
    info['category'] = None
    col_lower = col.lower()
    
    for category, mapping in category_mapping.items():
        # Check keywords
        matched = False
        for keyword in mapping['keywords']:
            if keyword.lower() in col_lower:
                info['category'] = category
                matched = True
                break
        
        # Check regex patterns if no keyword match
        if not matched and mapping['patterns']:
            for pattern in mapping['patterns']:
                if re.search(pattern, col_lower, re.IGNORECASE):
                    info['category'] = category
                    matched = True
                    break
        
        if matched:
            break

# Collect uncategorized columns
uncategorized = []
for col, info in parameter_quality.items():
    if info['category'] is None and col not in ['group', 'subject_id', 'nr', 'Wiek', 'BMI kg/m2']:
        uncategorized.append(col)
        info['category'] = 'UNCATEGORIZED'
        info['notes'].append('REVIEW MANUALLY - no category matched')

# Display results
print("\n" + "=" * 80)
print("CATEGORIZATION RESULTS")
print("=" * 80)

category_counts = {}
for info in parameter_quality.values():
    cat = info['category']
    if cat:
        category_counts[cat] = category_counts.get(cat, 0) + 1

print("\nCategory counts (including UNCATEGORIZED):")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}")

# Display all columns with their categories
print("\n" + "=" * 80)
print("FULL CATEGORIZATION LIST")
print("=" * 80)

for i, (col, info) in enumerate(list(parameter_quality.items())):
    print(f"{i+1:3}. {col} -> {info['category']}")

# Display ONLY uncategorized columns (MUST REVIEW)
print("\n" + "=" * 80)
print(f"⚠️ UNCATEGORIZED COLUMNS TO REVIEW ({len(uncategorized)})")
print("=" * 80)

if uncategorized:
    for col in uncategorized:
        info = parameter_quality[col]
        print(f"  ❌ {col}")
        print(f"     N={info['n_nonnull']}, type={info['data_type']}")
        print(f"     Sample: {df[col].dropna().head(3).tolist()}")
        print()
else:
    print("✅ No uncategorized columns found!")

# Update parameter_quality with category info
print(f"\n✅ Updated parameter_quality with categories for {len([i for i in parameter_quality.values() if i['category']])} columns")
print(f"⚠️ {len(uncategorized)} columns need manual review")

STEP 4: CLASSIFY PARAMETERS INTO CLINICAL CATEGORIES

CATEGORIZATION RESULTS

Category counts (including UNCATEGORIZED):
  Insulin_Fasting: 13
  Urinalysis: 13
  Glucose_OGTT: 9
  MPV: 9
  Prolactin: 8
  Other: 6
  RBC: 6
  HE4_ROMA: 6
  Microbiology: 6
  Progesterone_17OHP: 5
  FSH: 5
  LH: 5
  RDW: 5
  PT_APTT: 4
  AST: 4
  Virology: 4
  HbA1c: 4
  Cortisol_Dexamethasone: 4
  MCH: 4
  TSH: 4
  Testosterone_Total: 4
  Ultrasound: 4
  Index: 3
  AMH: 3
  Androstenedione: 3
  Anti_TG: 3
  Glucose_Fasting: 3
  HBsAg: 3
  Electrolytes: 3
  Cortisol_8AM: 3
  Lymphocytes: 3
  Platelets: 3
  WBC: 3
  Neutrophils: 3
  SHBG: 3
  CA125: 2
  DHEAS: 2
  Estradiol: 2
  Ferritin: 2
  Hematocrit: 2
  Hemoglobin: 2
  MCV: 2
  Basophils: 2
  Eosinophils: 2
  Immature_Granulocytes: 2
  Monocytes: 2
  Calcium: 2
  ALT: 1
  ASO: 1
  Anti_HCV: 1
  Anti_TPO: 1
  CRP: 1
  Bilirubin: 1
  CEA: 1
  CA19_9: 1
  Cholesterol_Total: 1
  D_Dimer: 1
  FT3: 1
  FT4: 1
  Fibrinogen: 1
  Phosphate: 1
  GH: 1
  GGT: 1
 

### Manual Corrections to Parameter Categories

Following automatic classification, several columns required manual re‑categorization due to ambiguous naming or incorrect automated assignment. The following corrections were applied:

| Correction | Original Category | New Category | Reason |
|------------|-------------------|--------------|--------|
| IGF-1 | Insulin_Fasting | IGF1 | IGF-1 is a growth factor, not insulin |
| TK (CT) scans | AST | CT_Scan | Imaging studies, not liver enzymes |
| USG elastography | AST | Ultrasound | Imaging study, not liver enzyme |
| TRAb (antibodies) | TSH | Thyroid_Autoantibodies | Autoantibodies, not thyroid hormone |
| MCHC | MCH | MCHC | Distinct red cell index (mean corpuscular hemoglobin concentration) |
| NRBC (nucleated RBC) | RBC | NRBC | Nucleated red blood cells differ from mature RBCs |

After applying these corrections, the full categorization list was verified and confirmed.

In [ ]:
# MANUAL CORRECTIONS TO PARAMETER CATEGORIES
print("=" * 80)
print("MANUAL CORRECTIONS TO PARAMETER CATEGORIES")
print("=" * 80)

# ============================================================================
# 1. IGF-1 → IGF1 (intead of Insulin_Fasting)
# ============================================================================
igf1_columns = [
    'IGF-1 (insulinopodobny czynnik wzrostu 1) (IGF1_L)'
]

for col in igf1_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'IGF1'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to IGF1')
        print(f"✓ {col}: {old_category} → IGF1")

# ============================================================================
# 2. TK jamy brzusznej → CT_Scan (instead of AST)
# ============================================================================
ct_columns = [
    'TK jamy brzusznej i miednicy z kontrastem (nan)',
    'TK jamy brzusznej z kontrastem (nan)'
]

for col in ct_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'CT_Scan'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to CT_Scan')
        print(f"✓ {col}: {old_category} → CT_Scan")

# ============================================================================
# 3. USG wątroby elastografia → Ultrasound (instead of AST)
# ============================================================================
usg_columns = [
    'USG wątroby – elastografia (nan)'
]

for col in usg_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'Ultrasound'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to Ultrasound')
        print(f"✓ {col}: {old_category} → Ultrasound")

# ============================================================================
# 4. TRAb → Thyroid_Autoantibodies (instead of TSH)
# ============================================================================
trab_columns = [
    'P/c przeciw receptorowi TSH (TRAb) (O15) (TRAB)',
    'P/c przeciw receptorowi TSH (TRAb) (TRAB_2)'
]

for col in trab_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'Thyroid_Autoantibodies'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to Thyroid_Autoantibodies')
        print(f"✓ {col}: {old_category} → Thyroid_Autoantibodies")

# ============================================================================
# 5. MCHC → MCHC (osobna kategoria, instead of MCH)
# ============================================================================
mchc_columns = [
    'Morfologia CBC (MCHC)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCHC)'
]

for col in mchc_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'MCHC'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to MCHC')
        print(f"✓ {col}: {old_category} → MCHC")

# ============================================================================
# 6. NRBC → NRBC (osobna kategoria, instead of RBC)
# ============================================================================
nrbc_columns = [
    'Morfologia CBC (NRBC_B)',
    'Morfologia CBC (NRBC_P)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_P)'
]

for col in nrbc_columns:
    if col in parameter_quality:
        old_category = parameter_quality[col]['category']
        parameter_quality[col]['category'] = 'NRBC'
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: changed from {old_category} to NRBC')
        print(f"✓ {col}: {old_category} → NRBC")

# ============================================================================
# PODSUMOWANIE
# ============================================================================
print("\n" + "=" * 80)
print("MANUAL CORRECTIONS SUMMARY")
print("=" * 80)

# Wyświetl nowe kategorie dla poprawionych kolumn
print("\nCorrected columns and their new categories:")
all_corrected = igf1_columns + ct_columns + usg_columns + trab_columns + mchc_columns + nrbc_columns
for col in all_corrected:
    if col in parameter_quality:
        print(f"  {col[:60]}... → {parameter_quality[col]['category']}")



# Display all columns with their categories
print("\n" + "=" * 80)
print("FULL CATEGORIZATION LIST")
print("=" * 80)

for i, (col, info) in enumerate(list(parameter_quality.items())):
    print(f"{i+1:3}. {col} -> {info['category']}")

MANUAL CORRECTIONS TO PARAMETER CATEGORIES
✓ IGF-1 (insulinopodobny czynnik wzrostu 1) (IGF1_L): Insulin_Fasting → IGF1
✓ TK jamy brzusznej i miednicy z kontrastem (nan): AST → CT_Scan
✓ TK jamy brzusznej z kontrastem (nan): AST → CT_Scan
✓ USG wątroby – elastografia (nan): AST → Ultrasound
✓ P/c przeciw receptorowi TSH (TRAb) (O15) (TRAB): TSH → Thyroid_Autoantibodies
✓ P/c przeciw receptorowi TSH (TRAb) (TRAB_2): TSH → Thyroid_Autoantibodies
✓ Morfologia CBC (MCHC): MCH → MCHC
✓ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCHC): MCH → MCHC
✓ Morfologia CBC (NRBC_B): RBC → NRBC
✓ Morfologia CBC (NRBC_P): RBC → NRBC
✓ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B): RBC → NRBC
✓ Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_P): RBC → NRBC

MANUAL CORRECTIONS SUMMARY

Corrected columns and their new categories:
  IGF-1 (insulinopodobny czynnik wzrostu 1

***

In [ ]:
# STEP 5: PARAMETERS WITH VERY FEW DATA (Two Thresholds)
print("=" * 80)
print("STEP 5: PARAMETERS WITH VERY FEW DATA (Two Thresholds)")
print("=" * 80)

# Define thresholds
EXCLUSION_THRESHOLD = 10  # N < 10 -> exclude automatically
WARNING_THRESHOLD = 30    # 10 ≤ N < 30 -> manual verification needed

# Collect parameters by threshold
exclude_params = []      # N < 10
verify_params = []       # 10 ≤ N < 30
sufficient_params = []   # N ≥ 30

for col, info in parameter_quality.items():
    if col in ['group', 'subject_id', 'nr', 'Wiek', 'BMI kg/m2']:
        continue
    
    n = info['n_nonnull']
    
    if n == 0:
        exclude_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'N={n} - completely empty column (no data)')
        info['recommended_action'] = 'exclude'
        info['notes'].append(f'No non-null values at all')

    elif n < EXCLUSION_THRESHOLD:
        exclude_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'N={n} < {EXCLUSION_THRESHOLD} - automatic exclusion')
        info['recommended_action'] = 'exclude'
        info['notes'].append(f'Only {n} non-null values')
    elif n < WARNING_THRESHOLD:
        verify_params.append(col)
        info['too_few_data'] = True
        info['useful'] = False  # Not useful for main analysis, but keep for manual review
        info['exclude_reason'].append(f'N={n} between {EXCLUSION_THRESHOLD} and {WARNING_THRESHOLD} - needs manual verification')
        info['recommended_action'] = 'manual_verify'
        info['notes'].append(f'Only {n} non-null values - consider if useful for subgroup analysis')
    else:
        sufficient_params.append(col)

# Display results
print(f"\nThresholds:")
print(f"  ❌ EXCLUDE (N < {EXCLUSION_THRESHOLD}): {len(exclude_params)} parameters")
print(f"  ⚠️ VERIFY ({EXCLUSION_THRESHOLD} ≤ N < {WARNING_THRESHOLD}): {len(verify_params)} parameters")
print(f"  ✅ SUFFICIENT (N ≥ {WARNING_THRESHOLD}): {len(sufficient_params)} parameters")

# ============================================================================
# DETAILED LIST - COMPLETELY EMPTY COLUMNS (N = 0)
# ============================================================================
print("\n" + "=" * 80)
print("📭 COMPLETELY EMPTY COLUMNS (N = 0)")
print("=" * 80)

empty_cols = [col for col in exclude_params if parameter_quality[col]['n_nonnull'] == 0]

if empty_cols:
    for col in empty_cols:
        info = parameter_quality[col]
        print(f"  {col}")
        print(f"     Category: {info['category']}")
        print(f"     Reason: {info['exclude_reason'][-1]}")
else:
    print("  No completely empty columns found")

# ============================================================================
# DETAILED LIST - EXCLUDE (N < 10)
# ============================================================================
print("\n" + "=" * 80)
print(f"❌ EXCLUDE (N < {EXCLUSION_THRESHOLD}) - AUTOMATIC EXCLUSION")
print("=" * 80)

if exclude_params:
    for col in sorted(exclude_params, key=lambda x: parameter_quality[x]['n_nonnull']):
        info = parameter_quality[col]
        print(f"\n  {col}")
        print(f"     Category: {info['category']}")
        print(f"     N={info['n_nonnull']}, unique={info['n_unique']}")
        print(f"     Sample values: {df[col].dropna().head(3).tolist()}")
else:
    print("  No parameters in this category")

# ============================================================================
# DETAILED LIST - MANUAL VERIFY (10 ≤ N < 30)
# ============================================================================
print("\n" + "=" * 80)
print(f"⚠️  MANUAL VERIFY ({EXCLUSION_THRESHOLD} ≤ N < {WARNING_THRESHOLD})")
print("=" * 80)

if verify_params:
    # Group by category
    verify_by_category = {}
    for col in verify_params:
        cat = parameter_quality[col]['category']
        if cat not in verify_by_category:
            verify_by_category[cat] = []
        verify_by_category[cat].append((col, parameter_quality[col]['n_nonnull']))
    
    for cat, cols in sorted(verify_by_category.items(), key=lambda x: -len(x[1])):
        print(f"\n  {cat} ({len(cols)} parameters):")
        for col, n in sorted(cols, key=lambda x: x[1]):
            info = parameter_quality[col]
            print(f"     {col}: N={n}, unique={info['n_unique']}")
else:
    print("  No parameters in this category")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)

print(f"\n{'Category':<25} {'Exclude':<10} {'Verify':<10} {'Sufficient':<10} {'Total':<10}")
print("-" * 65)

# Get categories from all parameters
all_categories = set()
for info in parameter_quality.values():
    if info['category'] and info['category'] not in ['Index', 'UNCATEGORIZED']:
        all_categories.add(info['category'])

category_summary = {}
for cat in sorted(all_categories):
    exclude_count = sum(1 for col in exclude_params if parameter_quality[col].get('category') == cat)
    verify_count = sum(1 for col in verify_params if parameter_quality[col].get('category') == cat)
    sufficient_count = sum(1 for col in sufficient_params if parameter_quality[col].get('category') == cat and parameter_quality[col].get('category') == cat)
    total = exclude_count + verify_count + sufficient_count
    if total > 0:
        print(f"{cat:<25} {exclude_count:<10} {verify_count:<10} {sufficient_count:<10} {total:<10}")


## Manual Verification Decision Summary (Step 5)

**Decision:** Exclude all 17 parameters from the `verify` category.

Based on the review, all parameters with **N between 10 and 30** were excluded from further analysis. The rationale is documented below.

---

## Rationale by Category

| Category | Parameters (N) | Decision | Justification |
|---|---|---|---|
| Prolactin | MAKROPRL (14), PRL16 (23), PRL10 (25), PRL10PEG (25) | ❌ Exclude | Better alternatives exist: `PRL godz. 10:00 (PRL10)` for PCOS group, `PRL ng/ml` for Control group |
| FSH (OGTT) | L_FSH0, L_FSH30, L_FSH60 (15 each) | ❌ Exclude | Small N; FSH information available from `FSH (FSH)` (PCOS) and `FSH IU/l` (Control) |
| LH (OGTT) | LH0 (15), L_LH30 (15), L_LH60 (15) | ❌ Exclude | Small N; LH information available from `LH (LH)` (PCOS) and `LH IU/l` (Control) |
| HbA1c | HBA1C_1 (23), HBA1C_2 (23) | ❌ Exclude | Insufficient sample size for meaningful analysis |
| Anty-TG | Anty-TG (O18) (21) | ❌ Exclude | Other any-TG columns have more data |
| CRP | Białko C-reaktywne (CRP) (19) | ❌ Exclude | Insufficient sample size |
| Index | Dokument (NAZWA) (18) | ❌ Exclude | Index/metadata column, not analytical; small N anyway |
| HE4_ROMA | Test Roma (HE4) (10) | ❌ Exclude | Very small N (only 10) – insufficient for analysis |
| Beta_HCG | beta HCG (L_HCG) (21) | ❌ Exclude | Only column for this parameter but highly incomplete; insufficient for meaningful analysis |

---

## Final Count

| Status | Count |
|---|---|
| ✅ Keep (Sufficient N ≥ 30) | 125 |
| ❌ Exclude (N < 10) | 78 |
| ❌ Exclude (Manual verify, 10–30) | 17 |
| **Total columns processed** | **220** |

In [ ]:
print("=" * 80)
print("FLAGGING MANUAL VERIFY PARAMETERS AS EXCLUDED")
print("=" * 80)

# List of parameters to exclude from manual verify category
manual_exclude_list = [
    # Prolactin (4)
    'Procent odzysku prolaktyny po precypitacji (MAKROPRL)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL16)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL10)',
    'PRL 10:00, 10:00 po PEG, 16:00 (PRL10PEG)',
    
    # FSH (3)
    'FSH 0\' 30\' 60\' (L_FSH0)',
    'FSH 0\' 30\' 60\' (L_FSH30)',
    'FSH 0\' 30\' 60\' (L_FSH60)',
    
    # LH (3)
    'LH 0\' 30\' 60\' (LH0)',
    'LH 0\' 30\' 60\' (L_LH30)',
    'LH 0\' 30\' 60\' (L_LH60)',
    
    # HbA1c (2)
    'Hemoglobina glikowana (HBA1C_1)',
    'Hemoglobina glikowana (HBA1C_2)',
    
    # Triglycerides / Anti-TG (1)
    'Anty-TG (O18)',
    
    # CRP (1)
    'Białko C-reaktywne (CRP)',
    
    # Index (1)
    'Dokument (NAZWA)',
    
    # HE4_ROMA (1)
    'Test Roma (HE4)',
    
    # Beta_HCG (1)
    'beta HCG (L_HCG)'
]

# Update parameter_quality for each
for col in manual_exclude_list:
    if col in parameter_quality:
        info = parameter_quality[col]
        info['useful'] = False
        info['too_few_data'] = True
        info['exclude_reason'].append(f'Manual decision: excluded after review (N={info["n_nonnull"]} between 10-30)')
        info['recommended_action'] = 'exclude'
        info['notes'].append('Manual exclude decision - insufficient data or better alternatives exist')
        print(f"  ❌ {col} (N={info['n_nonnull']}) -> excluded")

# Verify they are now marked as excluded
print("\n" + "=" * 80)
print("VERIFICATION: Parameters marked as 'useful = False'")
print("=" * 80)

verify_count = sum(1 for col in manual_exclude_list if col in parameter_quality and not parameter_quality[col]['useful'])
print(f"\n✅ Successfully flagged {verify_count} / {len(manual_exclude_list)} parameters as excluded")

# Final count of useful parameters
final_useful = sum(1 for info in parameter_quality.values() if info.get('useful') == True and info.get('category') not in ['Index'])
print(f"\n📊 FINAL COUNT - Useful parameters for analysis: {final_useful}")

In [ ]:
# STEP 6: DETECT FORMATTING ERRORS (optimized for your data types)

print("=" * 80)
print("STEP 6: DETECT FORMATTING ERRORS")
print("=" * 80)

# Get list of useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

print(f"Checking {len(useful_params)} useful parameters for formatting issues...")

# Text null patterns to check for
null_patterns = ['null', 'NULL', 'NaN', 'nan', 'None', 'N/A', 'n/a', '?', '--', '\\N', 'NA', 'Na', 'missing', 'Missing']

formatting_issues = []

for col in useful_params:
    info = parameter_quality[col]
    errors_found = []
    sample = df[col].dropna()
    
    if len(sample) == 0:
        continue
    
    # ===== CHECK 1: Numeric columns (float64, int64) =====
    if df[col].dtype in ['float64', 'int64']:
        # Check for negative values
        negative_count = (sample < 0).sum()
        if negative_count > 0:
            errors_found.append(f'contains negative values - {negative_count} occurrences')
        
    
    # ===== CHECK 2: String columns (str) =====
    elif df[col].dtype == 'str' or df[col].dtype == 'object':
        sample_str = sample.astype(str)
        
        # Check for text null values
        text_null_count = sample_str.isin(null_patterns).sum()
        if text_null_count > 0:
            null_example = sample_str[sample_str.isin(null_patterns)].iloc[0]
            errors_found.append(f'contains text null values (e.g., "{null_example}") - {text_null_count} occurrences')
        
        # Check for commas (Polish decimal format)
        comma_count = sample_str.str.contains(',').sum()
        if comma_count > 0:
            errors_found.append(f'contains commas (Polish decimal) - {comma_count} occurrences')
        
        # Check for special characters
        special_chars = sample_str.str.contains(r'[><~]', na=False).sum()
        if special_chars > 0:
            errors_found.append(f'contains special characters (>, <, ~) - {special_chars} occurrences')
        
        # Check if column could be numeric (most values are numbers)
        numeric_count = 0
        text_count = 0
        for val in sample_str.head(50):
            try:
                float(str(val).replace(',', '.'))
                numeric_count += 1
            except:
                text_count += 1
        
        if numeric_count > 0 and text_count > 0:
            errors_found.append(f'mixed types: {numeric_count} numeric, {text_count} text in sample (consider converting)')
        
        # Check for placeholder values
        placeholder_count = sample_str.str.contains(r'^\?$|^\.\.\.$|^xxx$|^---$', na=False, regex=True).sum()
        if placeholder_count > 0:
            errors_found.append(f'contains placeholder values - {placeholder_count} occurrences')
    
    
    if errors_found:
        formatting_issues.append({
            'column': col,
            'category': info['category'],
            'dtype': str(df[col].dtype),
            'errors': errors_found,
            'sample': sample.head(5).tolist(),
            'n_nonnull': info['n_nonnull']
        })
        info['formatting_errors'] = True
        info['exclude_reason'].extend(errors_found)
        info['recommended_action'] = 'fix_formatting'
        info['notes'].append(f'Formatting errors: {", ".join(errors_found)}')
        print(f"\n⚠️ {col} [{info['category']}] (dtype={df[col].dtype})")
        for err in errors_found:
            print(f"     - {err}")

# Display summary
print("\n" + "=" * 80)
print("FORMATTING ISSUES SUMMARY")
print("=" * 80)

print(f"\nFound {len(formatting_issues)} columns with formatting issues out of {len(useful_params)} useful parameters")

if formatting_issues:
    print("\nColumns with formatting issues:")
    for issue in formatting_issues:
        print(f"  • {issue['column']} ({issue['category']}) - dtype={issue['dtype']}, N={issue['n_nonnull']}")
        for err in issue['errors']:
            print(f"      - {err}")



## Step 6: Formatting Errors – Decisions Needed

The following columns contain formatting issues that prevent automatic conversion to numeric type. **A decision is required** on how to handle the problematic values.

### Columns Requiring Data Cleaning Decisions

| Column | Category | N | Issue | Problematic Values |
|--------|----------|---|-------|-------------------|
| `Morfologia CBC (NRBC_B)` | NRBC | 44 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Morfologia CBC (NRBC_P)` | NRBC | 44 | Commas | Clean comma values only (no mixed formats observed) |
| `Morfologia krwi... (NRBC_B)` | NRBC | 901 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Morfologia krwi... (NRBC_P)` | NRBC | 902 | Commas + mixed format | 0,00 \| 0,01 (contains pipe character) |
| `Witamina D (WITD)` | Vitamin_D | 809 | Commas + special character | <3,00 \| 4,48 (contains < and \|) |

### Decisions Required

1. **For NRBC columns with `"0,00 | 0,01"`:**
   - Which value should be kept? (first? second? both? average?)
   - Should these be treated as missing?

2. **For Vitamin D with `"<3,00 | 4,48"`:**
   - How to handle the `<` (below detection limit)? (impute as 3.00? treat as missing? keep as categorical?)
   - Which value to keep? (first? second? both?)


In [ ]:
print("=" * 80)
print("STEP 6: DATA QUALITY ISSUES – DECISIONS NEEDED")
print("=" * 80)

# List of columns with formatting issues requiring decisions
columns_needing_decisions = {
    'Morfologia CBC (NRBC_B)': {
        'category': 'NRBC',
        'n': 44,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00 | 0,01', '0,00']
    },
    'Morfologia CBC (NRBC_P)': {
        'category': 'NRBC',
        'n': 44,
        'issue': 'contains commas (Polish decimal)',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_B)': {
        'category': 'NRBC',
        'n': 901,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NRBC_P)': {
        'category': 'NRBC',
        'n': 902,
        'issue': 'contains commas (Polish decimal) + problematic value "0,00 | 0,01"',
        'sample': ['0,00', '0,00', '0,00', '0,00', '0,00']
    },
    'Witamina D (25- hydroksywitamina D3) (WITD)': {
        'category': 'Vitamin_D',
        'n': 809,
        'issue': 'contains commas (Polish decimal) + contains special character "<"',
        'sample': ['23,80', '13,00', '41,80', '17,90', '18,90']
    }
}

# Update parameter_quality with notes
for col, details in columns_needing_decisions.items():
    if col in parameter_quality:
        info = parameter_quality[col]
        info['formatting_errors'] = True
        info['recommended_action'] = 'decision_needed'
        info['notes'].append(f'Data cleaning decision needed: {details["issue"]}')
        info['notes'].append(f'Problematic sample values: {details["sample"]}')
        print(f"\n❓ {col}")
        print(f"     Category: {details['category']}")
        print(f"     N={details['n']}")
        print(f"     Issue: {details['issue']}")
        print(f"     Sample: {details['sample']}")
        print(f"     → Decision needed on how to handle problematic values before numeric conversion")

In [ ]:
# STEP 7: PARAMETERS WITH NO VARIANCE (≤2 UNIQUE VALUES)

print("=" * 80)
print("STEP 7: PARAMETERS WITH NO VARIANCE (≤2 UNIQUE VALUES)")
print("=" * 80)

# Get list of useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

print(f"Checking {len(useful_params)} useful parameters for low variance...")

no_variance_params = []
low_variance_params = []  # exactly 2 unique values

for col in useful_params:
    info = parameter_quality[col]
    n_unique = info['n_unique']
    n_nonnull = info['n_nonnull']
    
    if n_nonnull == 0:
        continue
    
    if n_unique == 1:
        no_variance_params.append({
            'column': col,
            'category': info['category'],
            'n_unique': n_unique,
            'n_nonnull': n_nonnull,
            'constant_value': df[col].dropna().iloc[0] if n_nonnull > 0 else None
        })
        info['no_variance'] = True
        info['useful'] = False
        info['exclude_reason'].append(f'no variance - only 1 unique value (constant: {df[col].dropna().iloc[0]})')
        info['recommended_action'] = 'exclude'
        
    elif n_unique == 2:
        low_variance_params.append({
            'column': col,
            'category': info['category'],
            'n_unique': n_unique,
            'n_nonnull': n_nonnull,
            'unique_values': df[col].dropna().unique().tolist()
        })
        # Don't automatically exclude - these might be binary flags
        info['notes'].append(f'low variance - only 2 unique values: {df[col].dropna().unique().tolist()}')

# Display results
print(f"\n✅ Parameters with NO variance (1 unique value): {len(no_variance_params)}")
print(f"⚠️ Parameters with LOW variance (2 unique values): {len(low_variance_params)}")

# ============================================================================
# DETAILED LIST - NO VARIANCE (CONSTANT COLUMNS)
# ============================================================================
print("\n" + "=" * 80)
print("❌ NO VARIANCE (1 UNIQUE VALUE) - AUTOMATIC EXCLUSION")
print("=" * 80)

if no_variance_params:
    for param in no_variance_params:
        print(f"\n  {param['column']}")
        print(f"     Category: {param['category']}")
        print(f"     N={param['n_nonnull']}, unique values=1")
        print(f"     Constant value: {param['constant_value']}")
        print(f"     → Excluded from further analysis")
else:
    print("  No constant columns found")

# ============================================================================
# DETAILED LIST - LOW VARIANCE (2 UNIQUE VALUES) - FOR REVIEW
# ============================================================================
print("\n" + "=" * 80)
print("⚠️ LOW VARIANCE (2 UNIQUE VALUES) - MANUAL REVIEW")
print("=" * 80)

if low_variance_params:
    # Group by category
    low_var_by_category = {}
    for param in low_variance_params:
        cat = param['category']
        if cat not in low_var_by_category:
            low_var_by_category[cat] = []
        low_var_by_category[cat].append(param)
    
    for cat, params in sorted(low_var_by_category.items()):
        print(f"\n  {cat} ({len(params)} parameters):")
        for param in params:
            print(f"     • {param['column']}")
            print(f"       N={param['n_nonnull']}, unique values: {param['unique_values']}")
else:
    print("  No parameters with exactly 2 unique values found")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("STEP 7 SUMMARY")
print("=" * 80)

print(f"""
Parameters with NO variance (excluded automatically): {len(no_variance_params)}
Parameters with LOW variance (2 values) - review needed: {len(low_variance_params)}

Remaining useful parameters after Step 7: {len([col for col, info in parameter_quality.items() if info.get('useful') == True and info.get('category') not in ['Index']])}
""")


## Step 7: Parameters with No / Low Variance – Summary

### Results

| Category | Count | Action |
|----------|-------|--------|
| No variance (1 unique value) | 0 | – |
| Low variance (2 unique values) | 1 | Manual review |
| Normal variance (≥3 unique values) | 116 | Keep |

### Low Variance Parameter – Review Decision

| Column | Category | N | Unique Values | Decision |
|--------|----------|---|---------------|----------|
| `HBsAg (L_HBSAG)` | HBsAg | 1113 | `['Niereaktywny', 'ujemny']` | ✅ **Keep** – clinically meaningful binary variable |

### Final Status

- **Parameters remaining after Step 7:** 117
- **No automatic exclusions** in this step
- One parameter reviewed and confirmed as keep

In [ ]:
# Update parameter_quality for this column
if 'HBsAg (L_HBSAG)' in parameter_quality:
    parameter_quality['HBsAg (L_HBSAG)']['notes'].append(
        'Manual review: two unique values ("Niereaktywny" and "ujemny") both mean "negative". No positive cases detected. '
        'Column can be kept as binary flag or consolidated to a single value.'
    )
    parameter_quality['HBsAg (L_HBSAG)']['recommended_action'] = 'keep_or_consolidate'

In [ ]:
print("=" * 80)
print("REMAINING PARAMETERS AFTER STEP 7 (N = 117)")
print("=" * 80)

# Get all useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

# Define which keys to display
display_keys = [
    'original_name',
    'category',
    'data_type',
    'n_nonnull',
    'n_unique',
    'only_in_group',
    'too_few_data',
    'no_variance',
    'formatting_errors',
    'unit_unknown',
    'detected_unit',
    'recommended_action'
]

# Build dataframe
remaining_df = pd.DataFrame([
    {
        'parameter': col,
        'category': info.get('category', ''),
        'data_type': info.get('data_type', ''),
        'n_nonnull': info.get('n_nonnull', 0),
        'n_unique': info.get('n_unique', 0),
        'only_in_group': info.get('only_in_group', ''),
        'too_few_data': info.get('too_few_data', False),
        'no_variance': info.get('no_variance', False),
        'formatting_errors': info.get('formatting_errors', False),
        'unit_unknown': info.get('unit_unknown', False),
        'detected_unit': info.get('detected_unit', ''),
        'recommended_action': info.get('recommended_action', '')
    }
    for col, info in parameter_quality.items()
    if info.get('useful') == True and info.get('category') not in ['Index']
])

# Sort by category and parameter name
remaining_df = remaining_df.sort_values(['category', 'parameter'])

# Display the table
print(f"\n📊 Total: {len(remaining_df)} parameters remaining for analysis\n")

# Display as formatted table
print(remaining_df.to_string(index=False))


In [ ]:
print("=" * 80)
print("STEP 8: UNIT DETECTION FOR REMAINING PARAMETERS")
print("=" * 80)

# Get all useful parameters
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

print(f"Analyzing {len(useful_params)} parameters for unit detection...")

# ============================================================================
# UNIT DETECTION FUNCTIONS
# ============================================================================

def extract_unit_from_name(column_name):
    """Extract unit from column name if present (e.g., 'mg/dl', 'ug/dl', 'nmol/l')"""
    name_lower = column_name.lower()
    
    unit_patterns = {
        'mg/dL': ['mg/dl', 'mg per dl', 'mg/ dl'],
        'ug/dL': ['ug/dl', 'μg/dl', 'mcg/dl', 'ug/ dl'],
        'ng/mL': ['ng/ml', 'ng per ml', 'ng/ ml'],
        'nmol/L': ['nmol/l', 'nmol per l', 'nmol/ l'],
        'pmol/L': ['pmol/l', 'pmol per l'],
        'μU/mL': ['uu/ml', 'μu/ml', 'micro u/ml', 'iu/ml'],
        'mIU/L': ['miu/l', 'miu per l'],
        'g/L': ['g/l', 'gram per l'],
        'mg/L': ['mg/l', 'mg per l'],
        'μg/L': ['ug/l', 'μg/l', 'mcg/l'],
        'fL': ['fl', 'femtoliter'],
        '%': ['%', 'percent', 'procent'],
        '10^3/uL': ['10^3/ul', '10^3/ul', 'k/ul', 'thousand per ul'],
        '10^9/L': ['10^9/l', 'g/l']
    }
    
    for unit, patterns in unit_patterns.items():
        for pattern in patterns:
            if pattern in name_lower:
                return unit
    return None

def detect_unit_by_values(series, category, param_name):
    """Detect unit based on statistical values and typical reference ranges"""
    s = pd.to_numeric(series, errors='coerce').dropna()
    if len(s) < 5:
        return None
    
    med = s.median()
    mean_val = s.mean()
    min_val = s.min()
    max_val = s.max()
    
    # Category-specific unit detection
    if category == 'Glucose_Fasting':
        if med > 20:
            return 'mg/dL'
        elif 3 <= med <= 20:
            return 'mmol/L'
    
    elif category == 'SHBG':
        if 20 < med < 200:
            return 'nmol/L'
        elif 20000 < med < 200000:
            return 'pmol/L'
    
    elif category == 'Testosterone_Total':
        # Female testosterone reference ranges:
        # ng/mL: 0.15 - 0.7 (rarely up to 1.0)
        # nmol/L: 0.5 - 2.4 (rarely up to 3.0)
        # Conversion: 1 ng/mL = 3.47 nmol/L
        
        if 0.05 < med < 1:
            return 'ng/mL (likely)'
        elif 1 <= med < 15:
            return 'nmol/L (likely)'
        elif med >= 15:
            return 'pmol/L? (very high - check)'
        else:
            return None
    
    elif category == 'Insulin_Fasting':
        # Fasting insulin reference ranges:
        # μU/mL: 2-25 (normal), up to 50 in IR
        # pmol/L: 14-175 (normal), up to 350 in IR
        # Conversion: 1 μU/mL = 6.945 pmol/L
        
        if med < 2:
            return 'unclear (very low) - possibly μU/mL or pmol/L'
        elif 2 <= med < 14:
            return 'μU/mL (likely) - would be 14-97 pmol/L if pmol/L'
        elif 14 <= med < 25:
            # Ambiguous range: 14-25 could be either
            return 'AMBIGUOUS - could be μU/mL (normal) or pmol/L (normal)'
        elif 25 <= med <= 50:
            return 'μU/mL (likely - elevated) - would be 174-347 pmol/L if pmol/L'
        elif 50 < med <= 175:
            return 'pmol/L (likely - normal range) - would be >7.2 μU/mL if μU/mL'
        elif 175 < med <= 350:
            return 'pmol/L (likely - elevated) - would be >25 μU/mL if μU/mL'
        elif med > 350:
            return 'pmol/L (very high) or mIU/L'
        else:
            return None
    
    elif category == 'Cortisol_8AM' or category == 'Cortisol_11PM' or category == 'Cortisol_Dexamethasone':
        if 5 <= med <= 30:
            return 'μg/dL'
        elif 138 <= med <= 700:
            return 'nmol/L'
    
    elif category == 'Prolactin':
        if 4 <= med <= 50:
            return 'ng/mL'
        elif 80 <= med <= 1000:
            return 'μIU/mL'
    
    elif category == 'D_Dimer':
        if med < 1:
            return 'mg/L'
        elif 100 < med < 1000:
            return 'ng/mL'
    
    elif category == 'Vitamin_D':
        # Vitamin D reference ranges:
        # ng/mL: deficiency <20, insufficiency 20-29, normal 30-80, optimal 30-50
        # nmol/L: deficiency <50, insufficiency 50-75, normal 75-200, optimal 75-125
        # Conversion: 1 ng/mL = 2.5 nmol/L
        
        if med < 20:
            # Could be ng/mL (deficient) or nmol/L (severely deficient)
            if med < 8:
                return 'nmol/L (likely - severe deficiency)'
            else:
                return 'AMBIGUOUS - could be ng/mL (deficient) or nmol/L (insufficient)'
        elif 20 <= med < 30:
            # 20-29 ng/mL = 50-72.5 nmol/L
            return 'AMBIGUOUS - could be ng/mL (insufficient) or nmol/L (insufficient)'
        elif 30 <= med <= 50:
            # 30-50 ng/mL = 75-125 nmol/L (optimal range for both!)
            return 'AMBIGUOUS - optimal range for BOTH units (30-50 ng/mL OR 75-125 nmol/L)'
        elif 50 < med <= 80:
            # 51-80 ng/mL = 127-200 nmol/L
            return 'ng/mL (likely - high normal) - would be >200 nmol/L if nmol/L'
        elif 80 < med <= 100:
            return 'ng/mL (likely - elevated) - would be >200-250 nmol/L if nmol/L'
        elif 100 < med <= 250:
            return 'nmol/L (likely - normal to optimal) - would be >40 ng/mL if ng/mL'
        elif med > 250:
            return 'nmol/L (likely - high) - would be >100 ng/mL if ng/mL'
        else:
            return None
    
    elif category in ['LH', 'FSH']:
        if 1 <= med <= 20:
            return 'IU/L'
    
    elif category == 'TSH':
        if 0.3 <= med <= 5:
            return 'μIU/mL'
    
    elif category in ['FT3', 'FT4']:
        if category == 'FT3' and 2 <= med <= 8:
            return 'pmol/L'
        elif category == 'FT4' and 10 <= med <= 25:
            return 'pmol/L'
    
    elif category == 'CRP':
        if med < 300:
            return 'mg/L'
    
    elif category == 'HDL' or category == 'LDL' or category == 'Cholesterol_Total':
        if 20 <= med <= 200:
            return 'mg/dL'
        elif 1 <= med <= 5:
            return 'mmol/L'
    
    elif category == 'Triglycerides':
        if 50 <= med <= 300:
            return 'mg/dL'
        elif 0.5 <= med <= 3.5:
            return 'mmol/L'
    
    elif category == 'Hematocrit':
        if 30 <= med <= 50:
            return '%'
    
    elif category == 'Hemoglobin':
        if 10 <= med <= 18:
            return 'g/dL'
    
    elif category == 'WBC':
        # White Blood Cell count reference ranges:
        # 10^3/uL (or 10^9/L): 4.0 - 11.0 (normal)
        # 10^6/uL: would be 4,000 - 11,000 (too high)
        # 10^12/L: would be 0.004 - 0.011 (too low)
        
        if 3 <= med <= 15:
            return '10^3/uL (or 10^9/L) - typical range'
        elif med < 3:
            return '10^3/uL (likely - low) or 10^9/L'
        elif med > 15 and med < 100:
            return '10^3/uL (likely - elevated)'
        elif med > 100:
            return '10^3/uL? (very high) or different unit'
        else:
            return None

    elif category == 'RBC':
        # Red Blood Cell count reference ranges:
        # 10^6/uL: 4.2 - 5.4 (women), 4.7 - 6.1 (men)
        # 10^12/L: 4.2 - 5.4 (same numeric value!)
        # 10^9/L: would be 4,200 - 5,400 (too high)
        
        if 3.5 <= med <= 6.5:
            return '10^6/uL or 10^12/L (numerically identical)'
        elif med < 3.5:
            return '10^6/uL or 10^12/L (likely - low)'
        elif med > 6.5 and med < 10:
            return '10^6/uL or 10^12/L (likely - elevated)'
        elif med > 10:
            return '10^9/L? (unlikely) - check manually'
        else:
            return None

    elif category == 'Platelets':
        # Platelet count reference ranges:
        # 10^3/uL (or 10^9/L): 150 - 450 (normal)
        # 10^6/uL: would be 150,000 - 450,000 (too high)
        # 10^12/L: would be 0.15 - 0.45 (too low)
        
        if 100 <= med <= 500:
            return '10^3/uL (or 10^9/L) - typical range'
        elif med < 100:
            return '10^3/uL (likely - thrombocytopenia)'
        elif med > 500 and med < 1000:
            return '10^3/uL (likely - thrombocytosis)'
        elif med > 1000:
            return '10^3/uL? (very high) or 10^6/uL?'
        else:
            return None
    
    return None

def get_reference_range(category, unit=None):
    """Get reference range for a given category and unit"""
    ranges = {
        'Glucose_Fasting': {
            'mg/dL': (70, 100),
            'mmol/L': (3.9, 5.6)
        },
        'SHBG': {
            'nmol/L': (30, 120),
            'pmol/L': (30000, 120000)
        },
        'Testosterone_Total': {
            'ng/mL': (0.15, 0.7),
            'nmol/L': (0.5, 2.4),
            'pmol/L': (1.7, 8.3)  # Free testosterone? careful
        },
        'Testosterone_Free': {
            'ng/dL': (0.3, 1.9),
            'pg/mL': (3, 19),
            'pmol/L': (10, 65)
        },
        'Insulin_Fasting': {
            'μU/mL': (2, 25),
            'pmol/L': (14, 175)
        },
        'Cortisol_8AM': {
            'μg/dL': (5, 25),
            'nmol/L': (138, 690)
        },
        'Cortisol_11PM': {
            'μg/dL': (1, 10),
            'nmol/L': (28, 276)
        },
        'Cortisol_Dexamethasone': {
            'μg/dL': (0, 1.8),  # after 1mg dexamethasone suppression
            'nmol/L': (0, 50)
        },
        'Prolactin': {
            'ng/mL': (4, 23),
            'μIU/mL': (85, 500),
            'mIU/L': (85, 500)
        },
        'TSH': {
            'μIU/mL': (0.3, 4.5),
            'mIU/L': (0.3, 4.5)
        },
        'FT3': {
            'pmol/L': (2.5, 6.5),
            'pg/mL': (1.5, 4.0)
        },
        'FT4': {
            'pmol/L': (10, 25),
            'ng/dL': (0.8, 1.8)
        },
        'Vitamin_D': {
            'ng/mL': (30, 80),
            'nmol/L': (75, 200)
        },
        'HDL': {
            'mg/dL': (40, 60),
            'mmol/L': (1.0, 1.6)
        },
        'LDL': {
            'mg/dL': (50, 130),
            'mmol/L': (1.3, 3.4)
        },
        'Cholesterol_Total': {
            'mg/dL': (125, 200),
            'mmol/L': (3.2, 5.2)
        },
        'Triglycerides': {
            'mg/dL': (50, 150),
            'mmol/L': (0.6, 1.7)
        },
        'WBC': {
            '10^3/uL': (4.0, 11.0),
            '10^9/L': (4.0, 11.0)
        },
        'RBC': {
            '10^6/uL': (4.2, 5.4),
            '10^12/L': (4.2, 5.4)
        },
        'Platelets': {
            '10^3/uL': (150, 450),
            '10^9/L': (150, 450)
        },
        'Hemoglobin': {
            'g/dL': (12, 16),
            'mmol/L': (7.4, 9.9)
        },
        'Hematocrit': {
            '%': (36, 46)
        },
        'MCV': {
            'fL': (80, 100)
        },
        'MCH': {
            'pg': (27, 32)
        },
        'MCHC': {
            'g/dL': (32, 36)
        },
        'MPV': {
            'fL': (7, 11)
        },
        'RDW': {
            '%': (11.5, 14.5)
        },
        'CRP': {
            'mg/L': (0, 5),
            'mg/dL': (0, 0.5)
        },
        'D_Dimer': {
            'mg/L': (0, 0.5),
            'ng/mL': (0, 500)
        }
    }
    
    # If unit is specified, return that specific range
    if unit:
        return ranges.get(category, {}).get(unit, None)
    
    # Otherwise return all ranges for this category
    return ranges.get(category, {})

# ============================================================================
# RUN UNIT DETECTION
# ============================================================================
unit_detection_results = []

for col in useful_params:
    info = parameter_quality[col]
    category = info['category']
    series = df[col]
    
    # Skip if not numeric (for now)
    if series.dtype not in ['float64', 'int64']:
        unit_detection_results.append({
            'parameter': col,
            'category': category,
            'unit_from_name': None,
            'unit_from_values': None,
            'final_unit': 'NON_NUMERIC',
            'confidence': 'low',
            'notes': f'Column type: {series.dtype}'
        })
        continue
    
    # Try to extract unit from name
    unit_from_name = extract_unit_from_name(col)
    
    # Try to detect unit from values
    unit_from_values = detect_unit_by_values(series, category, col)
    
    # Determine final unit and confidence
    final_unit = None
    confidence = 'low'
    notes = []
    
    if unit_from_name and unit_from_values:
        if unit_from_name == unit_from_values:
            final_unit = unit_from_name
            confidence = 'high'
            notes.append(f'Name and value detection agree: {final_unit}')
        else:
            final_unit = unit_from_name  # Prefer name if explicitly stated
            confidence = 'medium'
            notes.append(f'Name suggests {unit_from_name}, values suggest {unit_from_values}')
    elif unit_from_name:
        final_unit = unit_from_name
        confidence = 'medium'
        notes.append(f'Unit extracted from column name: {final_unit}')
    elif unit_from_values:
        final_unit = unit_from_values
        confidence = 'medium'
        notes.append(f'Unit detected from values: {final_unit}')
    else:
        # Fallback: try to determine based on typical ranges
        med = series.median()
        if 0.1 < med < 1000:
            notes.append(f'Unit could not be determined. Median={med:.2f}')
        else:
            notes.append(f'Unit could not be determined. Values may need scaling.')
    
    # Store results
    info['detected_unit'] = final_unit
    info['unit_unknown'] = (final_unit is None or final_unit == 'NON_NUMERIC')
    
    unit_detection_results.append({
        'parameter': col,
        'category': category,
        'n_nonnull': info['n_nonnull'],
        'median': series.median() if series.dtype in ['float64', 'int64'] else None,
        'min': series.min() if series.dtype in ['float64', 'int64'] else None,
        'max': series.max() if series.dtype in ['float64', 'int64'] else None,
        'unit_from_name': unit_from_name,
        'unit_from_values': unit_from_values,
        'final_unit': final_unit if final_unit else 'UNKNOWN',
        'confidence': confidence,
        'notes': '; '.join(notes)
    })
    
    # Print progress for uncertain cases
    if confidence == 'low' and final_unit is None:
        print(f"⚠️ {col[:60]}... [{category}] - unit detection failed")

# ============================================================================
# DISPLAY RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("UNIT DETECTION RESULTS")
print("=" * 80)

# Create results DataFrame
unit_df = pd.DataFrame(unit_detection_results)

# Display summary counts
print("\nUnit detection summary:")
print(unit_df['final_unit'].value_counts().to_string())

# Display parameters with UNKNOWN units
unknown_units = unit_df[unit_df['final_unit'] == 'UNKNOWN']
if len(unknown_units) > 0:
    print(f"\n⚠️ Parameters with UNKNOWN units ({len(unknown_units)}):")
    for _, row in unknown_units.iterrows():
        print(f"  • {row['parameter'][:60]}... [{row['category']}] - median={row['median']:.2f}")

# Display parameters with HIGH confidence
high_confidence = unit_df[unit_df['confidence'] == 'high']
if len(high_confidence) > 0:
    print(f"\n✅ Parameters with HIGH confidence unit detection ({len(high_confidence)}):")
    for _, row in high_confidence.iterrows():
        print(f"  • {row['parameter'][:60]}... -> {row['final_unit']}")

# Display full table
print("\n" + "=" * 80)
print("FULL UNIT DETECTION TABLE")
print("=" * 80)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 50)
print(unit_df[['parameter', 'category', 'final_unit', 'confidence', 'notes']].to_string(index=False))

# Update parameter_quality
for col, info in parameter_quality.items():
    if col in unit_df['parameter'].values:
        row = unit_df[unit_df['parameter'] == col].iloc[0]
        if row['final_unit'] != 'UNKNOWN' and row['final_unit'] != 'NON_NUMERIC':
            info['detected_unit'] = row['final_unit']
            info['unit_unknown'] = False

## Unit Identification – Manual Resolution

### Context

Automated unit detection was performed for 117 parameters using two approaches:
1. **Extraction from column names** (e.g., `ng/mL`, `mg/dL`, `nmol/L`)
2. **Value-based detection** (comparing medians to reference ranges)

However, several factors prevented fully automated identification:

| Issue | Examples |
|-------|----------|
| Overlapping reference ranges | Insulin (14–25 could be μU/mL or pmol/L), Vitamin D (30–50 could be ng/mL or nmol/L) |
| No units in column names | Cortisol, DHEAS, Estradiol – units must be inferred from values |
| Reference ranges difficult to predict in PCOS patients | Testosterone (elevated in PCOS – may exceed typical female ranges), Insulin (may be significantly elevated due to insulin resistance)|

### Decision

Due to the limitations above, **automated unit detection was abandoned** in favor of **manual assignment** based on:
- Clinical knowledge of standard reference ranges
- Expected values for women with and without PCOS
- Consistency with published literature

### Outcome

All 117 parameters were manually assigned their correct units. For any ambiguous cases (e.g., insulin columns with mixed units), unit conversion will be applied prior to analysis to ensure consistency.

> **Note:** Parameters flagged as `NON_NUMERIC` (e.g., HBsAg, NRBC, Vitamin D before cleaning) were excluded from unit detection and will be handled separately.

In [ ]:
print("=" * 80)
print("MANUAL UNIT ASSIGNMENT FOR REMAINING PARAMETERS")
print("=" * 80)

# Manual unit assignments for all UNKNOWN parameters
manual_units = {
    # Progesterone / 17-OHP (ng/mL - typical for Synacthen test)
    '17 - OH progesteron (L79) (17-OHPG)': 'ng/mL',
    '17 OH progesteron (L79)': 'ng/mL',
    'Test z Synacthenem (L79_0)': 'ng/mL',
    'Test z Synacthenem (L79_1)': 'ng/mL',
    'Test z Synacthenem (L79_2)': 'ng/mL',
    
    # Liver enzymes (U/L)
    'ALAT (ALT)': 'U/L',
    'ASPAT (AST)': 'U/L',
    'Gamma Glutamylotranspeptydaza (GGTP)': 'U/L',
    
    # AMH (ng/mL)
    'AMH (hormon anty-Mullerowski) (AMH_CP)': 'ng/mL',
    'AMH-anty Mullerian Hormon (AMH)': 'ng/mL',
    
    # Androstenedione (ng/mL)
    'Androstendion (ANDRO)': 'ng/mL',
    'Androstendion (I31)': 'ng/mL',
    'Androstendion (I31) (ANDRO)': 'ng/mL',
    
    # Anti-thyroglobulin antibodies (IU/mL)
    'Anty-TG (p/c przeciw tyreoglobulinie) (ATG)': 'IU/mL',
    'P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)': 'IU/mL',
    
    # Anti-TPO (IU/mL)
    'Anty-TPO (ATA_TPO)': 'IU/mL',
    
    # Bilirubin (mg/dL)
    'Bilirubina całkowita (TBIL)': 'mg/dL',
    
    # CA125 (U/mL)
    'Ca125 (CA125)': 'U/mL',
    
    # DHEAS (μg/dL)
    'DHEAS (DHEA)': 'μg/dL',
    
    # Estradiol (pg/mL)
    'Estradiol (ESTRA)': 'pg/mL',
    'Estradiol (L_ESTRA)': 'pg/mL',
    
    # FT4 (ng/dL - note: 1.18 is typical for ng/dL, not pmol/L)
    'FT4 (FT4)': 'ng/dL',
    
    # Creatinine (mg/dL)
    'Kreatynina (KREAT)': 'mg/dL',
    
    # Glucose OGTT (mg/dL)
    'Krzywa cukrowa - 2 punktowa (GLU120)': 'mg/dL',
    
    # Electrolytes (mmol/L)
    'Jonogram (Sód, potas, chlorki) (L_CI)': 'mmol/L',
    'Jonogram (Sód, potas, chlorki) (L_K)': 'mmol/L',
    'Jonogram (Sód, potas, chlorki) (L_NA)': 'mmol/L',
    
    # Cortisol (μg/dL)
    'Kortyzol godz. 23:00 (KOR23)': 'μg/dL',
    'Kortyzol po Dexamethasonie (KORD)': 'μg/dL',
    'Kortyzol po Dexamethasonie [1mg] (KORD)': 'μg/dL',
    'Kortyzol po Dexamethasonie [2mg] (KOR2DXM)': 'μg/dL',
    
    # CBC indices
    'Morfologia CBC (MCH)': 'pg',
    'Morfologia CBC (MCHC)': 'g/dL',
    'Morfologia CBC (MCV)': 'fL',
    'Morfologia CBC (MPV)': 'fL',
    'Morfologia CBC (PCT)': '%',
    'Morfologia CBC (PDW)': 'fL',
    'Morfologia CBC (PLCR)': '%',
    'Morfologia CBC (RDW)': '%',
    'Morfologia CBC (RDWSD)': 'fL',
    
    # CBC differential - absolute counts (10^3/uL)
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASO)': '10^3/uL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (EOS#)': '10^3/uL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (LIM#)': '10^3/uL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MON#)': '10^3/uL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT#)': '10^3/uL',
    
    # CBC differential - percentages (%)
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASOB)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (EOS)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (LIM)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MON)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT)': '%',
    
    # CBC differential - indices (same as CBC)
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCH)': 'pg',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCHC)': 'g/dL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCV)': 'fL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MPV)': 'fL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PCT)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PDW)': 'fL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLCR)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDW)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDWSD)': 'fL',
    
    # Immature granulocytes
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (IG_B)': '10^3/uL',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (IG_P)': '%',
}

# Apply manual assignments
assigned_count = 0
not_found = []

for col_name, unit in manual_units.items():
    # Try to find exact match or partial match
    matched_col = None
    for actual_col in parameter_quality.keys():
        if actual_col == col_name:
            matched_col = actual_col
            break
        elif col_name in actual_col:
            matched_col = actual_col
            break
    
    if matched_col:
        old_unit = parameter_quality[matched_col].get('detected_unit', 'UNKNOWN')
        parameter_quality[matched_col]['detected_unit'] = unit
        parameter_quality[matched_col]['unit_unknown'] = False
        parameter_quality[matched_col]['notes'].append(f'MANUAL UNIT ASSIGNMENT: {old_unit} → {unit}')
        assigned_count += 1
        print(f"✓ {matched_col[:55]}... → {unit}")
    else:
        not_found.append(col_name)

print(f"\n" + "=" * 80)
print(f"MANUAL UNIT ASSIGNMENT SUMMARY")
print("=" * 80)
print(f"✅ Assigned: {assigned_count} parameters")
print(f"⚠️ Not found: {len(not_found)} parameters")

if not_found:
    print("\nParameters not found (may have different names):")
    for col in not_found[:20]:
        print(f"  • {col}")

# Verify remaining UNKNOWN units after manual assignment
remaining_unknown = [
    col for col, info in parameter_quality.items() 
    if info.get('unit_unknown', True) and info.get('detected_unit') == 'UNKNOWN'
    and info.get('category') not in ['Index']
]

print(f"\n📊 Remaining UNKNOWN after manual assignment: {len(remaining_unknown)}")

# Final summary of all units
print("\n" + "=" * 80)
print("FINAL UNIT SUMMARY AFTER MANUAL ASSIGNMENT")
print("=" * 80)

unit_summary = {}
for col, info in parameter_quality.items():
    unit = info.get('detected_unit')
    if unit and unit != 'UNKNOWN' and unit != 'NON_NUMERIC':
        unit_summary[unit] = unit_summary.get(unit, 0) + 1

for unit, count in sorted(unit_summary.items(), key=lambda x: -x[1]):
    print(f"  {unit}: {count} parameters")

In [ ]:
print("=" * 80)
print("CHECK UNITS FOR REMAINING PARAMETERS")
print("=" * 80)

# Get all useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

# Define which keys to display
display_keys = [
    'original_name',
    'category',
    'data_type',
    'n_nonnull',
    'n_unique',
    'only_in_group',
    'too_few_data',
    'no_variance',
    'formatting_errors',
    'unit_unknown',
    'detected_unit',
    'recommended_action'
]

# Build dataframe
remaining_df = pd.DataFrame([
    {
        'parameter': col,
        'category': info.get('category', ''),
        'data_type': info.get('data_type', ''),
        'n_nonnull': info.get('n_nonnull', 0),
        'n_unique': info.get('n_unique', 0),
        'only_in_group': info.get('only_in_group', ''),
        'too_few_data': info.get('too_few_data', False),
        'no_variance': info.get('no_variance', False),
        'formatting_errors': info.get('formatting_errors', False),
        'unit_unknown': info.get('unit_unknown', False),
        'detected_unit': info.get('detected_unit', ''),
        'recommended_action': info.get('recommended_action', '')
    }
    for col, info in parameter_quality.items()
    if info.get('useful') == True and info.get('category') not in ['Index']
])

# Sort by category and parameter name
remaining_df = remaining_df.sort_values(['category', 'parameter'])

# Display the table
print(f"\n📊 Total: {len(remaining_df)} parameters remaining for analysis\n")

# Display as formatted table
print(remaining_df.to_string(index=False))


In [ ]:
# Get useful parameters (numeric only)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True 
    and info.get('category') not in ['Index']
    and df[col].dtype in ['float64', 'int64']  # Only numeric columns
]

print("=" * 80)
print("DESCRIPTION OF USEFUL PARAMETERS BY CATEGORY")
print("=" * 80)

# Group by category
categories = {}
for col in useful_params:
    cat = parameter_quality[col].get('category', 'Unknown')
    if cat not in categories:
        categories[cat] = []
    categories[cat].append(col)

# Display description for each category
for cat, cols in sorted(categories.items()):
    print(f"\n{'=' * 80}")
    print(f"CATEGORY: {cat} ({len(cols)} parameters)")
    print('=' * 80)
    
    if cols:
        # Get description and transpose
        desc_df = df[cols].describe().T
        
        # Add detected unit as first column after parameter name
        desc_df.insert(0, 'detected_unit', [parameter_quality[col].get('detected_unit', 'Unknown') for col in desc_df.index])
        
        # Add group info (PCOS vs Control_no_PCOS)
        group_info = []
        for col in desc_df.index:
            only_in_group = parameter_quality[col].get('only_in_group')
            if only_in_group == 'PCOS':
                group_info.append('PCOS only')
            elif only_in_group == 'Control_no_PCOS':
                group_info.append('Control only')
            else:
                group_info.append('both groups')
        desc_df.insert(1, 'group', group_info)
        
        # Reorder columns
        cols_order = ['detected_unit', 'group', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
        desc_df = desc_df[[c for c in cols_order if c in desc_df.columns]]
        
        # Display
        print(desc_df.to_string())
    else:
        print("  No parameters in this category")

# Save to CSV (numeric only)
all_desc_list = []
for col in useful_params:
    info = parameter_quality[col]
    only_in_group = info.get('only_in_group')
    group_status = 'PCOS only' if only_in_group == 'PCOS' else ('Control only' if only_in_group == 'Control_no_PCOS' else 'both groups')
    
    row = {
        'parameter': col,
        'category': info.get('category', 'Unknown'),
        'detected_unit': info.get('detected_unit', 'Unknown'),
        'group': group_status,
        'count': df[col].count(),
        'mean': df[col].mean(),
        'std': df[col].std(),
        'min': df[col].min(),
        '25%': df[col].quantile(0.25),
        '50%': df[col].median(),
        '75%': df[col].quantile(0.75),
        'max': df[col].max()
    }
    all_desc_list.append(row)

if all_desc_list:
    all_desc_df = pd.DataFrame(all_desc_list)
    all_desc_df.to_csv('useful_parameters_description.csv', index=False)
    print("\n✅ Saved to 'useful_parameters_description.csv'")
else:
    print("\n⚠️ No numeric parameters found")

# Also list non-numeric useful parameters separately (for completeness)
non_numeric_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True 
    and info.get('category') not in ['Index']
    and df[col].dtype not in ['float64', 'int64']
]

if non_numeric_params:
    print("\n" + "=" * 80)
    print("NON-NUMERIC USEFUL PARAMETERS (excluded from description)")
    print("=" * 80)
    for col in non_numeric_params:
        info = parameter_quality[col]
        only_in_group = info.get('only_in_group')
        group_status = 'PCOS only' if only_in_group == 'PCOS' else ('Control only' if only_in_group == 'Control_no_PCOS' else 'both groups')
        print(f"  • {col}")
        print(f"    Category: {info.get('category')}")
        print(f"    Group: {group_status}")
        print(f"    Dtype: {df[col].dtype}")
        print(f"    Unique values: {df[col].nunique()}")
        print()

Androstendion jest poza normą dla zdrowych kobiet, natomiast w PCOS jest on podwyszony, oc się pokrywa
BASO, powinniśmy na odwrót przypisać liczbę bezwzględną i procent
Estradiol, trzeba dojść do tego, co oznacza L_ESTRA
SKOŃCZYŁEM ANALIZĘ NA GGTP
Insulin_Fasting całe będzie w jednostce μU/mL
CATEGORY: RBC (2 parameters) - 10^6/uL bedzie poprawne
Testosteron (L_TESTOS) i testosteron ng/ml ustawiamy jednostke ng/mL 
Testosteron wolny (O41) (TEST-F) i Wolny testosteron (O41.W) ustawiamy jednostke nmol/L
WBC bedzie 10^3/uL 
Platelets 10^3/uL


## Manual Unit Corrections

Automated unit detection flagged several parameters with `UNKNOWN` units or ambiguous assignments. The following manual corrections were applied based on clinical knowledge, expected reference ranges, and laboratory conventions.

### Correction Details

#### 1. Basophil Parameters (BASO / BASOB) – Swapped Units

**Issue:** Automated detection incorrectly assigned `BASO` as absolute count and `BASOB` as percentage. Based on naming conventions (`#` or `B` suffix typically denotes absolute count), these were swapped.

| Parameter | Original Unit | Corrected Unit | Rationale |
|-----------|---------------|----------------|-----------|
| `BASO` | `10^3/uL` | `%` | No `#` or `B` suffix → percentage |
| `BASOB` | `%` | `10^3/uL` | `B` suffix → absolute count |

#### 2. Insulin – Unified Units (μU/mL)

**Issue:** Insulin measurements were recorded in mixed units across different columns (μU/mL and pmol/L), making cross-column comparison impossible.

**Action:** All 12 insulin columns were standardized to **μU/mL**.

| Column Type | Original Unit | Corrected Unit | Note |
|-------------|---------------|----------------|------|
| Fasting insulin (INSUL, INSUL_0, L97_0m, L97_0, insulina uU/ml) | μU/mL | μU/mL | Already correct |
| Post-OGTT insulin (L97_30m, L97_60m, L97_120m, L97_1, L97_2) | μU/mL | μU/mL | Already correct |
| Post-OGTT insulin (INSUL_1, INSUL_2) | pmol/L | μU/mL | **Requires division by 6.945** |

> **Important:** Values in `INSUL_1` and `INSUL_2` must be converted from pmol/L to μU/mL (÷ 6.945) before analysis.

#### 3. Red Blood Cells (RBC) – Unit Standardization

**Issue:** RBC values are numerically identical in `10^6/uL` and `10^12/L`, making automated distinction impossible.

**Action:** Both RBC columns were manually assigned to **`10^6/uL`** (the conventional clinical unit).

| Parameter | Corrected Unit |
|-----------|----------------|
| `Morfologia CBC (RBC)` | `10^6/uL` |
| `Morfologia krwi... (RBC)` | `10^6/uL` |

#### 4. Testosterone – Distinguishing Total vs Free

**Issue:** Total and free testosterone were initially grouped together with ambiguous units.

**Action:** 

| Parameter | Type | Corrected Unit |
|-----------|------|----------------|
| `Testosteron (L_TESTOS)` | Total | `ng/mL` |
| `testosteron ng/ml` | Total | `ng/mL` |
| `Testosteron wolny (TEST-F)` | Free | `nmol/L` |
| `Wolny testosteron (O41.W)` | Free | `nmol/L` |

#### 5. White Blood Cells (WBC) – Unit Standardization

**Action:** All WBC columns assigned to **`10^3/uL`** (conventional clinical unit).

#### 6. Platelets – Unit Standardization

**Action:** All platelet columns assigned to **`10^3/uL`** (conventional clinical unit).

#### 7. Micro Prefix Normalization (ug/dL → μg/dL)

**Issue:** Inconsistent notation of micro-units (`ug/dL` vs `μg/dL`) introduces redundant unit representations.

**Action:** All `ug/dL` units were standardized to **μg/dL**.

| Original Unit | Corrected Unit |
|---------------|----------------|
| `ug/dL`       | `μg/dL`        |


In [ ]:
print("=" * 80)
print("MANUAL UNIT CORRECTIONS BASED ON CLINICAL REVIEW")
print("=" * 80)


# ============================================================================
# 1. BASO / BASOB – odwrócone jednostki (poprawiamy)
# ============================================================================
baso_corrections = {
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASO)': '%',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (BASOB)': '10^3/uL'
}
for col, unit in baso_corrections.items():
    if col in parameter_quality:
        old = parameter_quality[col].get('detected_unit', 'Unknown')
        parameter_quality[col]['detected_unit'] = unit
        parameter_quality[col]['unit_unknown'] = False
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: BASO/BASOB swapped. {old} → {unit}')
        print(f"✓ {col[:55]}... {old} → {unit}")


# ============================================================================
# 2. Insulin_Fasting – WSZYSTKIE kolumny do μU/mL
# ============================================================================
insulin_columns_to_unify = [
    'Insulina (INSUL)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_0)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_1)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_2)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)',
    'insulina uU/ml'
]
for col in insulin_columns_to_unify:
    if col in parameter_quality:
        old = parameter_quality[col].get('detected_unit', 'Unknown')
        parameter_quality[col]['detected_unit'] = 'μU/mL'
        parameter_quality[col]['unit_unknown'] = False
        parameter_quality[col]['notes'].append(f'MANUAL CORRECTION: unified to μU/mL')
        print(f"✓ {col[:55]}... {old} → μU/mL ")

# ============================================================================
# 3. RBC – 10^6/uL (poprawne)
# ============================================================================
for col in [
        'Morfologia CBC (RBC)', 
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RBC)'
    ]:
    actual_col = next((c for c in parameter_quality if col in c), None)
    if actual_col:
        parameter_quality[actual_col]['detected_unit'] = '10^6/uL'
        parameter_quality[actual_col]['notes'].append('MANUAL CORRECTION: set to 10^6/uL')
        print(f"✓ {actual_col[:50]}... → 10^6/uL")

# ============================================================================
# 4. Testosterone_Total – ng/mL dla całkowitego, nmol/L dla wolnego
# ============================================================================
if 'Testosteron (L_TESTOS)' in parameter_quality:
    parameter_quality['Testosteron (L_TESTOS)']['detected_unit'] = 'ng/mL'
    print("✓ Testosteron (L_TESTOS) → ng/mL")

if 'testosteron ng/ml' in parameter_quality:
    parameter_quality['testosteron ng/ml']['detected_unit'] = 'ng/mL'
    print("✓ testosteron ng/ml → ng/mL")

for col in ['Testosteron wolny (O41) (TEST-F)', 'Wolny testosteron (O41.W)']:
    if col in parameter_quality:
        parameter_quality[col]['detected_unit'] = 'nmol/L'
        print(f"✓ {col} → nmol/L")

# ============================================================================
# 5. WBC – 10^3/uL
# ============================================================================
for col in ['Morfologia CBC (WBC)', 'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (WBC)', 'WBC 10^3/uL']:
    actual_col = next((c for c in parameter_quality if col in c), None)
    if actual_col:
        parameter_quality[actual_col]['detected_unit'] = '10^3/uL'
        print(f"✓ {actual_col[:50]}... → 10^3/uL")

# ============================================================================
# 6. Platelets – 10^3/uL
# ============================================================================
for col in ['Morfologia CBC (PLT)', 'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLT)', 'PLT 10^3/uL']:
    actual_col = next((c for c in parameter_quality if col in c), None)
    if actual_col:
        parameter_quality[actual_col]['detected_unit'] = '10^3/uL'
        print(f"✓ {actual_col[:50]}... → 10^3/uL")
        
# ============================================================================
# 7. FIX: ug/dL → μg/dL (normalizacja mikro-prefiksu)
# ============================================================================
for col, info in parameter_quality.items():
    unit = info.get('detected_unit', None)
    
    if unit is None:
        continue

    if isinstance(unit, str) and unit.lower() == 'ug/dl':
        old = unit
        parameter_quality[col]['detected_unit'] = 'μg/dL'
        parameter_quality[col]['notes'].append('MANUAL CORRECTION: ug/dL → μg/dL (micro prefix normalized)')
        print(f"✓ {col[:55]}... {old} → μg/dL")

# ============================================================================
# PODSUMOWANIE
# ============================================================================
print("\n" + "=" * 80)
print("SUMMARY OF UNIT CORRECTIONS")
print("=" * 80)

# Count units after corrections
unit_counts = {}
for info in parameter_quality.values():
    unit = info.get('detected_unit', 'UNKNOWN')
    if unit and unit != 'UNKNOWN' and unit != 'NON_NUMERIC':
        # Simplify unit strings for counting
        if 'μU/mL' in unit:
            unit = 'μU/mL'
        unit_counts[unit] = unit_counts.get(unit, 0) + 1

print("\nFinal unit distribution:")
for unit, count in sorted(unit_counts.items(), key=lambda x: -x[1]):
    print(f"  {unit}: {count} parameters")

print("\n✅ All unit corrections applied. Ready for next steps.")

In [ ]:
print("=" * 80)
print("CHECK UNITS FOR REMAINING PARAMETERS")
print("=" * 80)

# Get all useful parameters (not excluded)
useful_params = [
    col for col, info in parameter_quality.items() 
    if info.get('useful') == True and info.get('category') not in ['Index']
]

# Define which keys to display
display_keys = [
    'original_name',
    'category',
    'data_type',
    'n_nonnull',
    'n_unique',
    'only_in_group',
    'too_few_data',
    'no_variance',
    'formatting_errors',
    'unit_unknown',
    'detected_unit',
    'recommended_action'
]

# Build dataframe
remaining_df = pd.DataFrame([
    {
        'parameter': col,
        'category': info.get('category', ''),
        'data_type': info.get('data_type', ''),
        'n_nonnull': info.get('n_nonnull', 0),
        'n_unique': info.get('n_unique', 0),
        'only_in_group': info.get('only_in_group', ''),
        'too_few_data': info.get('too_few_data', False),
        'no_variance': info.get('no_variance', False),
        'formatting_errors': info.get('formatting_errors', False),
        'unit_unknown': info.get('unit_unknown', False),
        'detected_unit': info.get('detected_unit', ''),
        'recommended_action': info.get('recommended_action', '')
    }
    for col, info in parameter_quality.items()
    if info.get('useful') == True and info.get('category') not in ['Index']
])

# Sort by category and parameter name
remaining_df = remaining_df.sort_values(['category', 'parameter'])

# Display the table
print(f"\n📊 Total: {len(remaining_df)} parameters remaining for analysis\n")

# Display as formatted table
print(remaining_df.to_string(index=False))


## 9. Column consolidation – merging parameters across groups and methods

### Overview

Multiple columns often measured the same clinical parameter across different patient subgroups (PCOS vs Control), laboratory methods (CBC vs CBC with differential), or OGTT protocols (3‑point vs 4‑point). This step consolidates such columns into single representative variables, removes redundant parameters, and selects the most appropriate variable for downstream analysis.

### Cross‑Group and Cross‑Method Column Merging

All columns measuring the same biological parameter were merged into a single variable. No patient overlap was detected between any of the consolidated columns, confirming that they originate from distinct patient subgroups and can be safely combined.

#### Merging Groups


| Group | Original Columns | Unit | Action |
|-------|------------------|------|--------|
| **AMH** | `AMH (hormon anty-Mullerowski) (AMH_CP)` (PCOS) + `AMH-anty Mullerian Hormon (AMH)` (PCOS) + `AMH ng/ml` (Control) | ng/mL | Merge → `AMH_ng_ml` |
| **Androstenedione** | `Androstendion (ANDRO)` (PCOS) + `Androstendion (I31)` (PCOS) + `Androstendion (I31) (ANDRO)` (PCOS) | ng/mL | Merge → `androstenedione_ng_ml` |
| **Anti-TG** | `P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)` (PCOS) + `Anty-TG (p/c przeciw tyreoglobulinie) (ATG)` (PCOS) | IU/mL | Merge → `anti_tg_IU_mL` |
| **Cortisol_8AM** | `Kortyzol godz. 08:00 (KOR)` (PCOS) + `kortyzol 8.00 ug/dl` (Control) | μg/dL | Merge → `cortisol_8am_ug_dL` |
| **DHEAS** | `DHEAS (DHEA)` (PCOS) + `DHEAS ug/dl` (Control) | μg/dL | Merge → `dheas_ug_dL` |
| **FSH** | `FSH IU/l` (Control) + `FSH (FSH)` (PCOS) | IU/L | Merge → `fsh_iu_l` |
| **Glucose_Fasting** | `Krzywa cukrowa - 2 punktowa (L_GLU_0)` (PCOS) + `glu 0' mg/dl` (Control) | mg/dL | Merge → `glucose_fasting_mg_dL` |
| **Hematocrit** | `Morfologia CBC (HCT)` (PCOS) + `Morfologia krwi... (HCT)` (PCOS) | % | Merge → `hematocrit_percent` |
| **Hemoglobin** | `Morfologia CBC (HGB)` (PCOS) + `Morfologia krwi... (HGB)` (PCOS) | g/dL | Merge → `hemoglobin_g_dL` |
| **Insulin (fasting)** | `Insulina (INSUL)` (PCOS) + `Insulina po 75g glukozy (3pkt.) (INSUL_0)` (PCOS) + `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)` (PCOS) + `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)` (PCOS) + `insulina uU/ml` (Control) | μU/mL | Merge → `insulin_fasting_uU_ml` |
| **Insulin (30 min)** | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)` (PCOS) + `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)` (PCOS) | μU/mL | Merge → `insulin_30min_uU_ml` |
| **Insulin (60 min, 3‑point OGTT)** | `Insulina po 75g glukozy (3pkt.) (INSUL_1)` (PCOS) | μU/mL | Keep as → `insulin_60min_3pt_uU_ml` |
| **Insulin (60 min, 4‑point OGTT)** | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)` (PCOS) + `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)` (PCOS) | μU/mL | Merge → `insulin_60min_4pt_uU_ml` |
| **Insulin (120 min, 3‑point OGTT)** | `Insulina po 75g glukozy (3pkt.) (INSUL_2)` (PCOS) | μU/mL | Keep as → `insulin_120min_3pt_uU_ml` |
| **Insulin (120 min, 4‑point OGTT)** | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)` (PCOS) | μU/mL | Keep as → `insulin_120min_4pt_uU_ml` |
| **LH** | `LH (LH)` (PCOS) + `LH IU/l` (Control) | IU/L | Merge → `lh_iu_l` |
| **Lymphocytes (absolute)** | `Limf 10^3/uL` (Control) + `Morfologia krwi... (LIM#)` (PCOS) | 10^3/uL | Merge → `lymphocytes_abs_10_3_uL` |
| **MCH** | `Morfologia CBC (MCH)` (PCOS) + `Morfologia krwi... (MCH)` (PCOS) | pg | Merge → `mch_pg` |
| **MCHC** | `Morfologia CBC (MCHC)` (PCOS) + `Morfologia krwi... (MCHC)` (PCOS) | g/dL | Merge → `mchc_g_dL` |
| **MCV** | `Morfologia CBC (MCV)` (PCOS) + `Morfologia krwi... (MCV)` (PCOS) | fL | Merge → `mcv_fL` |
| **MPV** | `MPV fL` (Control) + `Morfologia CBC (MPV)` (PCOS) + `Morfologia krwi... (MPV)` (PCOS) | fL | Merge → `mpv_fL` |
| **PCT** | `Morfologia CBC (PCT)` (PCOS) + `Morfologia krwi... (PCT)` (PCOS) | % | Merge → `pct_percent` |
| **PDW** | `Morfologia CBC (PDW)` (PCOS) + `Morfologia krwi... (PDW)` (PCOS) | fL | Merge → `pdw_fL` |
| **PLCR** | `Morfologia CBC (PLCR)` (PCOS) + `Morfologia krwi... (PLCR)` (PCOS) | % | Merge → `plcr_percent` |
| **Neutrophils (absolute)** | `Neu 10^3/uL` (Control) + `Morfologia krwi... (NEUT#)` (PCOS) | 10^3/uL | Merge → `neutrophils_abs_10_3_uL` |
| **Platelets** | `PLT 10^3/uL` (Control) + `Morfologia CBC (PLT)` (PCOS) + `Morfologia krwi... (PLT)` (PCOS) | 10^3/uL | Merge → `platelets_10_3_uL` |
| **17‑OHP (baseline)** | `17 - OH progesteron (L79) (17-OHPG)` (PCOS) + `17 OH progesteron (L79)` (PCOS) | ng/mL | Merge → `17ohp_baseline_ng_ml` |
| **Prolactin** | `PRL godz. 10:00 (PRL10)` (PCOS) + `PRL ng/ml` (Control) | ng/mL | Merge → `prolactin_ng_ml` |
| **RBC** | `Morfologia CBC (RBC)` (PCOS) + `Morfologia krwi... (RBC)` (PCOS) | 10^6/uL | Merge → `rbc_10_6_uL` |
| **RDW** | `Morfologia CBC (RDW)` (PCOS) + `Morfologia krwi... (RDW)` (PCOS) + `RDW %` (Control) | % | Merge → `rdw_percent` |
| **RDWSD** | `Morfologia CBC (RDWSD)` (PCOS) + `Morfologia krwi... (RDWSD)` (PCOS) | fL | Merge → `rdwsd_fL` |
| **SHBG** | `SHBG nmol/l` (Control) + `SHBG (L_SHGB)` (PCOS) | nmol/L | Merge → `shbg_nmol_l` |
| **Testosterone (total)** | `testosteron ng/ml` (Control) + `Testosteron (L_TESTOS)` (PCOS) | ng/mL | Merge → `testosterone_ng_ml` |
| **Testosterone (free)** | `Testosteron wolny (O41) (TEST-F)` (PCOS) + `Wolny testosteron (O41.W)` (PCOS) | nmol/L | Merge → `testosterone_free_nmol_L` |
| **TSH** | `TSH (TSH)` (PCOS) + `TSH uIU/ml` (Control) | μIU/mL | Merge → `tsh_uIU_mL` |
| **WBC** | `Morfologia CBC (WBC)` (PCOS) + `Morfologia krwi... (WBC)` (PCOS) + `WBC 10^3/uL` (Control) | 10^3/uL | Merge → `wbc_10_3_uL` |


estradiolu nie łączymy, bo za due rónice w średnich, a to parametr mierzony w rónych okresach u kobiet

In [ ]:
# Sprawdź overlap między dwiema kolumnami AMH w PCOS
overlap_amh = df[df['AMH (hormon anty-Mullerowski) (AMH_CP)'].notna() & df['AMH-anty Mullerian Hormon (AMH)'].notna()]
print(f"Pacjentki z oboma pomiarami AMH: {len(overlap_amh)}")

if len(overlap_amh) > 0:
    # Sprawdź korelację
    corr = overlap_amh['AMH (hormon anty-Mullerowski) (AMH_CP)'].corr(
           overlap_amh['AMH-anty Mullerian Hormon (AMH)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnami ANDRO w PCOS
# Kolumny ANDRO
andro_columns = [
    'Androstendion (ANDRO)',
    'Androstendion (I31)',
    'Androstendion (I31) (ANDRO)'
]

# Sprawdź overlap dla każdej pary
for i, col1 in enumerate(andro_columns):
    for col2 in andro_columns[i+1:]:
        if col1 in df.columns and col2 in df.columns:
            overlap = df[df[col1].notna() & df[col2].notna()]
            print(f"\n📊 Overlap: {col1} vs {col2}")
            print(f"   Pacjentki z oboma pomiarami: {len(overlap)}")
            
            if len(overlap) > 0:
                corr = overlap[col1].corr(overlap[col2])
                print(f"   Korelacja: {corr:.4f}")
                
                # Dodatkowe statystyki
                mean1 = overlap[col1].mean()
                mean2 = overlap[col2].mean()
                print(f"   Średnia ({col1[:20]}...): {mean1:.3f}")
                print(f"   Średnia ({col2[:20]}...): {mean2:.3f}")
                print(f"   Różnica średnich: {abs(mean1 - mean2):.3f}")

# Sprawdź trzykolumnowy overlap (wszystkie trzy naraz)
overlap_all = df[df['Androstendion (ANDRO)'].notna() & 
                 df['Androstendion (I31)'].notna() & 
                 df['Androstendion (I31) (ANDRO)'].notna()]

print(f"\n📊 THREE‑WAY OVERLAP (wszystkie trzy kolumny):")
print(f"   Pacjentki z wszystkimi trzema pomiarami: {len(overlap_all)}")

if len(overlap_all) > 0:
    # Macierz korelacji
    corr_matrix = overlap_all[andro_columns].corr()
    print("\n   Macierz korelacji:")
    print(corr_matrix.to_string())
    
    # Średnie
    print("\n   Średnie wartości:")
    for col in andro_columns:
        print(f"     {col}: {overlap_all[col].mean():.3f}")
df[['Androstendion (ANDRO)', 'Androstendion (I31)', 'Androstendion (I31) (ANDRO)']].describe()


## Androstenedione – Column Consolidation

### Decision

Three columns measuring androstenedione were merged into a single variable: `androstenedione_ng_ml`.

| Column | Group | N | Mean (ng/mL) |
|--------|-------|---|--------------|
| `Androstendion (ANDRO)` | PCOS | 682 | 1.61 |
| `Androstendion (I31)` | PCOS | 189 | 3.37 |
| `Androstendion (I31) (ANDRO)` | PCOS | 255 | 2.65 |

### Rationale

| Criterion | Assessment |
|-----------|------------|
| **No patient overlap** | Zero patients had measurements in more than one column → columns come from different testing batches / clinical scenarios, not conflicting measurements |
| **Clinically plausible range** | All values fall within biologically expected ranges for PCOS (normal female: 0.5–3.0 ng/mL; PCOS often elevated to 3–10 ng/mL) |
| **Unit consistency** | Values <10 ng/mL strongly indicate all three columns are in **ng/mL** (not nmol/L, which would be <10 nmol/L = ~2.9 ng/mL) |
| **Pooling benefit** | Merging increases total sample size from individual columns (N=682, 189, 255) to a combined **N=1,126** non‑null values |

### Conclusion

The three columns represent the same biological parameter measured in different patient subgroups with no overlapping cases. Consolidation preserves statistical power while maintaining clinical validity.

In [ ]:
# Sprawdź overlap między dwiema kolumnami anty-tg w PCOS
overlap_atg = df[df['P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)'].notna() & df['Anty-TG (p/c przeciw tyreoglobulinie) (ATG)'].notna()]
print(f"Pacjentki z oboma pomiarami anty-Tg: {len(overlap_atg)}")

if len(overlap_atg) > 0:
    # Sprawdź korelację
    corr = overlap_atg['P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)'].corr(
           overlap_atg['Anty-TG (p/c przeciw tyreoglobulinie) (ATG)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnamie hct w PCOS
overlap_hct = df[df['Morfologia CBC (HCT)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HCT)'].notna()]
print(f"Pacjentki z oboma pomiarami hct: {len(overlap_hct)}")

if len(overlap_hct) > 0:
    # Sprawdź korelację
    corr = overlap_hct['Morfologia CBC (HCT)'].corr(
           overlap_hct['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HCT)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnamie hgb w PCOS
overlap_hgb = df[df['Morfologia CBC (HGB)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HGB)'].notna()]
print(f"Pacjentki z oboma pomiarami hgb: {len(overlap_hgb)}")

if len(overlap_hgb) > 0:
    # Sprawdź korelację
    corr = overlap_hgb['Morfologia CBC (HGB)'].corr(
           overlap_hgb['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HGB)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

## Insulin Column Consolidation – Summary

New insulin parameters will be created. Each new variable consolidates columns measuring the same time point.

---

### Fasting Insulin (0 min) → `insulin_fasting_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina (INSUL)` | 744 |
| 2 | `Insulina po 75g glukozy (3pkt.) (INSUL_0)` | 222 |
| 3 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)` | 83 |
| 4 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)` | 62 |
| 5 | `insulina uU/ml` | 45 |

✅ No patient overlap detected – values from different columns are mutually exclusive. The new variable takes the value from the highest‑priority column available for each patient.

---

### Insulin 30 min → `insulin_30min_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)` | 83 |
| 2 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)` | 61 |

✅ No patient overlap – columns are mutually exclusive.

---

### Insulin 60 min – separate protocol variables (no merge across protocols)

**3‑point protocol** → `insulin_60min_3pt_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina po 75g glukozy (3pkt.) (INSUL_1)` | 221 |

**4‑point protocol** → `insulin_60min_4pt_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)` | 83 |
| 2 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)` | 62 |

Within the 4‑point protocol, no patient overlap between `L97_60m` and `L97_2`.

---

### Insulin 120 min – separate protocol variables

**3‑point protocol** → `insulin_120min_3pt_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina po 75g glukozy (3pkt.) (INSUL_2)` | 221 |

**4‑point protocol** → `insulin_120min_4pt_uU_ml`

| Priority | Column | N |
|----------|--------|---|
| 1 | `Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)` | 83 |

---

### Summary of New Variables

| New Variable | Time Point | Protocol | N (total non‑null) |
|--------------|------------|----------|-------------------|
| `insulin_fasting_uU_ml` | 0 min | all | 744 + 222 + 83 + 62 + 45 → **1,156** |
| `insulin_30min_uU_ml` | 30 min | 4‑point | 83 + 61 → **144** |
| `insulin_60min_3pt_uU_ml` | 60 min | 3‑point | **221** |
| `insulin_60min_4pt_uU_ml` | 60 min | 4‑point | 83 + 62 → **145** |
| `insulin_120min_3pt_uU_ml` | 120 min | 3‑point | **221** |
| `insulin_120min_4pt_uU_ml` | 120 min | 4‑point | **83** |

In [ ]:
print("=" * 80)
print("INSULIN COLUMNS – DESCRIPTIVE STATISTICS")
print("=" * 80)

# List of insulin columns
insulin_columns = [
    'Insulina (INSUL)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_0)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_1)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_2)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)',
    'insulina uU/ml'
]

# Create description dataframe
insulin_desc = []

for col in insulin_columns:
    if col in df.columns:
        series = df[col].dropna()
        insulin_desc.append({
            'Column': col,
            'N': len(series),
            'Mean': series.mean(),
            'Std': series.std(),
            'Min': series.min(),
            '25%': series.quantile(0.25),
            'Median': series.median(),
            '75%': series.quantile(0.75),
            'Max': series.max()
        })

insulin_desc_df = pd.DataFrame(insulin_desc)

# Display
print(insulin_desc_df.to_string(index=False))

# Save to CSV
insulin_desc_df.to_csv('insulin_columns_description.csv', index=False)
print("\n✅ Saved to 'insulin_columns_description.csv'")

In [ ]:
print("=" * 80)
print("INSULIN COLUMNS – OVERLAP CHECK FOR SAME TIME POINTS")
print("=" * 80)

# ============================================================================
# 1. FASTING INSULIN (0 min) – wszystkie kolumny mierzące insulinę na czczo
# ============================================================================
fasting_columns = [
    'Insulina (INSUL)',
    'Insulina po 75g glukozy (3pkt.) (INSUL_0)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)',
    'insulina uU/ml'
]

print("\n" + "=" * 80)
print("1. FASTING INSULIN (0 min) – OVERLAP CHECK")
print("=" * 80)

# Sprawdź pacjentki z więcej niż jedną kolumną insuliny na czczo
fasting_overlap_count = 0
fasting_overlap_samples = []

for i, col1 in enumerate(fasting_columns):
    if col1 not in df.columns:
        continue
    for col2 in fasting_columns[i+1:]:
        if col2 not in df.columns:
            continue
        overlap = df[df[col1].notna() & df[col2].notna()]
        if len(overlap) > 0:
            fasting_overlap_count += len(overlap)
            fasting_overlap_samples.append({
                'col1': col1,
                'col2': col2,
                'overlap_count': len(overlap),
                'sample_values': overlap[[col1, col2]].head(5).values.tolist()
            })
            print(f"\n⚠️ Overlap: {col1[:50]}... vs {col2[:50]}...")
            print(f"   Liczba pacjentek: {len(overlap)}")
            print(f"   Przykładowe pary wartości:")
            for idx, (val1, val2) in enumerate(overlap[[col1, col2]].head(5).values):
                print(f"     {idx+1}: {val1} vs {val2}")

if fasting_overlap_count == 0:
    print("\n✅ BRAK overlapu między kolumnami insuliny na czczo – można bezpiecznie scalić")

# ============================================================================
# 2. INSULIN 30 min (L97_30m vs L97_1)
# ============================================================================
print("\n" + "=" * 80)
print("2. INSULIN 30 min – OVERLAP CHECK")
print("=" * 80)

col_30min = ['Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)', 'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)']
if col_30min[0] in df.columns and col_30min[1] in df.columns:
    overlap_30 = df[df[col_30min[0]].notna() & df[col_30min[1]].notna()]
    if len(overlap_30) > 0:
        print(f"\n⚠️ Overlap: {len(overlap_30)} pacjentek")
        print(f"   Przykładowe wartości:")
        for idx, (val1, val2) in enumerate(overlap_30[col_30min].head(5).values):
            print(f"     {idx+1}: {val1} vs {val2}")
    else:
        print("\n✅ BRAK overlapu – można bezpiecznie scalić")

# ============================================================================
# 3. INSULIN 60 min (INSUL_1 vs L97_60m vs L97_2) – z różnych protokołów
# ============================================================================
print("\n" + "=" * 80)
print("3. INSULIN 60 min – OVERLAP CHECK (między protokołami)")
print("=" * 80)

col_60min = [
    'Insulina po 75g glukozy (3pkt.) (INSUL_1)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)'
]

for i, col1 in enumerate(col_60min):
    if col1 not in df.columns:
        continue
    for col2 in col_60min[i+1:]:
        if col2 not in df.columns:
            continue
        overlap = df[df[col1].notna() & df[col2].notna()]
        if len(overlap) > 0:
            print(f"\n⚠️ Overlap: {col1[:50]}... vs {col2[:50]}...")
            print(f"   Liczba pacjentek: {len(overlap)}")
            for idx, (val1, val2) in enumerate(overlap[[col1, col2]].head(5).values):
                print(f"     {idx+1}: {val1} vs {val2}")
        else:
            print(f"\n✅ BRAK overlapu: {col1[:40]}... vs {col2[:40]}...")

# ============================================================================
# 4. INSULIN 120 min (INSUL_2 vs L97_120m)
# ============================================================================
print("\n" + "=" * 80)
print("4. INSULIN 120 min – OVERLAP CHECK (między protokołami)")
print("=" * 80)

col_120min = [
    'Insulina po 75g glukozy (3pkt.) (INSUL_2)',
    'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)'
]

if col_120min[0] in df.columns and col_120min[1] in df.columns:
    overlap_120 = df[df[col_120min[0]].notna() & df[col_120min[1]].notna()]
    if len(overlap_120) > 0:
        print(f"\n⚠️ Overlap: {len(overlap_120)} pacjentek")
        print(f"   Przykładowe wartości:")
        for idx, (val1, val2) in enumerate(overlap_120[col_120min].head(5).values):
            print(f"     {idx+1}: {val1} vs {val2}")
    else:
        print("\n✅ BRAK overlapu")


In [ ]:
# Sprawdź overlap między dwiema kolumnamie mch w PCOS
overlap_mch = df[df['Morfologia CBC (MCH)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCH)'].notna()]
print(f"Pacjentki z oboma pomiarami mch: {len(overlap_mch)}")

if len(overlap_mch) > 0:
    # Sprawdź korelację
    corr = overlap_mch['Morfologia CBC (MCH)'].corr(
           overlap_mch['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCH)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnamie mchc w PCOS
overlap_mchc = df[df['Morfologia CBC (MCHC)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCHC)'].notna()]
print(f"Pacjentki z oboma pomiarami mchc: {len(overlap_mchc)}")

if len(overlap_mch) > 0:
    # Sprawdź korelację
    corr = overlap_mch['Morfologia CBC (MCH)'].corr(
           overlap_mch['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCH)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnamie mcv w PCOS
overlap_mcv = df[df['Morfologia CBC (MCV)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCV)'].notna()]
print(f"Pacjentki z oboma pomiarami mcv: {len(overlap_mcv)}")

if len(overlap_mcv) > 0:
    # Sprawdź korelację
    corr = overlap_mcv['Morfologia CBC (MCV)'].corr(
           overlap_mcv['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCV)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

## Platelet Parameters – Column Consolidation

No overlap was detected between any of the platelet parameter columns, confirming they come from different patient subgroups. All columns within each parameter were merged into a single variable.

| Parameter | Merged Columns | Unit | New Variable |
|-----------|----------------|------|--------------|
| **MPV** (mean platelet volume) | `MPV fL` (Control) + `Morfologia CBC (MPV)` (PCOS) + `Morfologia krwi... (MPV)` (PCOS) | fL | `mpv_fL` |
| **PCT** (plateletcrit) | `Morfologia CBC (PCT)` (PCOS) + `Morfologia krwi... (PCT)` (PCOS) | % | `pct_percent` |
| **PDW** (platelet distribution width) | `Morfologia CBC (PDW)` (PCOS) + `Morfologia krwi... (PDW)` (PCOS) | fL | `pdw_fL` |
| **PLCR** (platelet large cell ratio) | `Morfologia CBC (PLCR)` (PCOS) + `Morfologia krwi... (PLCR)` (PCOS) | % | `plcr_percent` |

For each parameter, the differential column (`Morfologia krwi...`) was prioritized due to larger sample size, followed by the CBC column, followed by Control column where applicable.

In [ ]:
print("=" * 80)
print("PLATELET PARAMETERS – OVERLAP CHECK BETWEEN CBC AND DIFFERENTIAL")
print("=" * 80)

# ============================================================================
# 1. MPV (mean platelet volume)
# ============================================================================
print("\n" + "=" * 80)
print("1. MPV (mean platelet volume) – OVERLAP CHECK")
print("=" * 80)

mpv_columns = [
    ('MPV fL', 'Control'),
    ('Morfologia CBC (MPV)', 'PCOS_CBC'),
    ('Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MPV)', 'PCOS_diff')
]

for i, (col1, name1) in enumerate(mpv_columns):
    if col1 not in df.columns:
        continue
    for col2, name2 in mpv_columns[i+1:]:
        if col2 not in df.columns:
            continue
        overlap = df[df[col1].notna() & df[col2].notna()]
        if len(overlap) > 0:
            print(f"\n⚠️ Overlap: {col1} vs {col2}")
            print(f"   Liczba pacjentek: {len(overlap)}")
            if len(overlap) > 0:
                corr = overlap[col1].corr(overlap[col2])
                print(f"   Korelacja: {corr:.3f}")
        else:
            print(f"\n✅ BRAK overlapu: {col1} vs {col2}")

# ============================================================================
# 2. PCT (plateletcrit)
# ============================================================================
print("\n" + "=" * 80)
print("2. PCT (plateletcrit) – OVERLAP CHECK")
print("=" * 80)

pct_columns = [
    'Morfologia CBC (PCT)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PCT)'
]

if pct_columns[0] in df.columns and pct_columns[1] in df.columns:
    overlap_pct = df[df[pct_columns[0]].notna() & df[pct_columns[1]].notna()]
    if len(overlap_pct) > 0:
        print(f"\n⚠️ Overlap: {len(overlap_pct)} pacjentek")
        corr = overlap_pct[pct_columns[0]].corr(overlap_pct[pct_columns[1]])
        print(f"   Korelacja: {corr:.3f}")
    else:
        print("\n✅ BRAK overlapu")

# ============================================================================
# 3. PDW (platelet distribution width)
# ============================================================================
print("\n" + "=" * 80)
print("3. PDW (platelet distribution width) – OVERLAP CHECK")
print("=" * 80)

pdw_columns = [
    'Morfologia CBC (PDW)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PDW)'
]

if pdw_columns[0] in df.columns and pdw_columns[1] in df.columns:
    overlap_pdw = df[df[pdw_columns[0]].notna() & df[pdw_columns[1]].notna()]
    if len(overlap_pdw) > 0:
        print(f"\n⚠️ Overlap: {len(overlap_pdw)} pacjentek")
        corr = overlap_pdw[pdw_columns[0]].corr(overlap_pdw[pdw_columns[1]])
        print(f"   Korelacja: {corr:.3f}")
    else:
        print("\n✅ BRAK overlapu")

# ============================================================================
# 4. PLCR (platelet large cell ratio)
# ============================================================================
print("\n" + "=" * 80)
print("4. PLCR (platelet large cell ratio) – OVERLAP CHECK")
print("=" * 80)

plcr_columns = [
    'Morfologia CBC (PLCR)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLCR)'
]

if plcr_columns[0] in df.columns and plcr_columns[1] in df.columns:
    overlap_plcr = df[df[plcr_columns[0]].notna() & df[plcr_columns[1]].notna()]
    if len(overlap_plcr) > 0:
        print(f"\n⚠️ Overlap: {len(overlap_plcr)} pacjentek")
        corr = overlap_plcr[plcr_columns[0]].corr(overlap_plcr[plcr_columns[1]])
        print(f"   Korelacja: {corr:.3f}")
    else:
        print("\n✅ BRAK overlapu")


In [ ]:
# Sprawdź overlap między dwiema kolumnamie neutrophils w PCOS
overlap_neutrophils = df[df['Neu 10^3/uL'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT#)'].notna()]
print(f"Pacjentki z oboma pomiarami neutrophils: {len(overlap_neutrophils)}")

if len(overlap_neutrophils) > 0:
    # Sprawdź korelację
    corr = overlap_neutrophils['Neu 10^3/uL'].corr(
           overlap_neutrophils['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT#)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

In [ ]:
# Sprawdź overlap między dwiema kolumnamie plt w PCOS
overlap_plt = df[df['Morfologia CBC (PLT)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLT)'].notna()]
print(f"Pacjentki z oboma pomiarami plt: {len(overlap_plt)}")

if len(overlap_plt) > 0:
    # Sprawdź korelację
    corr = overlap_plt['Morfologia CBC (PLT)'].corr(
           overlap_plt['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLT)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

## 17‑OH Progesterone – Column Consolidation

### Decision

Only the two columns with sufficient sample size were merged. The remaining columns will likely be excluded in later steps due to limited data.

| Group | Original Columns | Unit | Action |
|-------|------------------|------|--------|
| **17‑OHP (baseline)** | `17 - OH progesteron (L79) (17-OHPG)` (PCOS, N=965) + `17 OH progesteron (L79)` (PCOS, N=212) | ng/mL | Merge → `17ohp_baseline_ng_ml` |

### Excluded from merge

| Column | N | Reason |
|--------|---|--------|
| `Test z Synacthenem (L79_0)` | 52 | Different measurement phase (baseline but from Synacthen protocol) |
| `Test z Synacthenem (L79_1)` | 54 | Post‑stimulation 1st phase |
| `Test z Synacthenem (L79_2)` | 54 | Post‑stimulation 2nd phase |

> These columns may represent different physiological phases (baseline within Synacthen test vs. true baseline) and will be evaluated separately in subsequent steps.

In [ ]:
print("=" * 80)
print("17-OH PROGESTERONE COLUMNS – DESCRIPTIVE STATISTICS")
print("=" * 80)

# List of 17-OHP columns
ohp_columns = [
    '17 - OH progesteron (L79) (17-OHPG)',
    '17 OH progesteron (L79)',
    'Test z Synacthenem (L79_0)',
    'Test z Synacthenem (L79_1)',
    'Test z Synacthenem (L79_2)'
]

# Create description dataframe
ohp_desc = []

for col in ohp_columns:
    if col in df.columns:
        series = df[col].dropna()
        ohp_desc.append({
            'Column': col,
            'N': len(series),
            'Mean': series.mean(),
            'Std': series.std(),
            'Min': series.min(),
            '25%': series.quantile(0.25),
            'Median': series.median(),
            '75%': series.quantile(0.75),
            'Max': series.max()
        })

ohp_desc_df = pd.DataFrame(ohp_desc)

# Display
print(ohp_desc_df.to_string(index=False))

# Save to CSV
ohp_desc_df.to_csv('ohp_columns_description.csv', index=False)
print("\n✅ Saved to 'ohp_columns_description.csv'")

In [ ]:
print("=" * 80)
print("17-OHP BASELINE COLUMNS – OVERLAP CHECK")
print("=" * 80)

# Baseline columns (pre-stimulation)
baseline_cols = [
    '17 - OH progesteron (L79) (17-OHPG)',
    '17 OH progesteron (L79)',
    'Test z Synacthenem (L79_0)'
]

# Check overlap between each pair
for i, col1 in enumerate(baseline_cols):
    if col1 not in df.columns:
        print(f"\n⚠️ {col1} not found in dataframe")
        continue
    for col2 in baseline_cols[i+1:]:
        if col2 not in df.columns:
            print(f"\n⚠️ {col2} not found in dataframe")
            continue
        overlap = df[df[col1].notna() & df[col2].notna()]
        print(f"\n{'=' * 60}")
        print(f"Overlap: {col1[:40]}... vs {col2[:40]}...")
        print(f"Pacjentki z oboma pomiarami: {len(overlap)}")
        
        if len(overlap) > 0:
            corr = overlap[col1].corr(overlap[col2])
            print(f"Korelacja: {corr:.4f}")
            print(f"\nPrzykładowe pary wartości:")
            for idx, (val1, val2) in enumerate(overlap[[col1, col2]].head(5).values):
                print(f"  {idx+1}: {val1:.3f} vs {val2:.3f}")
        else:
            print("✅ BRAK overlapu – kolumny pochodzą z różnych podgrup pacjentek")

# Three-way overlap (all three baseline columns)
print("\n" + "=" * 80)
print("THREE-WAY OVERLAP (wszystkie trzy kolumny baseline)")
print("=" * 80)

if all(col in df.columns for col in baseline_cols):
    overlap_all = df[df[baseline_cols[0]].notna() & 
                     df[baseline_cols[1]].notna() & 
                     df[baseline_cols[2]].notna()]
    print(f"Pacjentki ze wszystkimi trzema pomiarami: {len(overlap_all)}")
    
    if len(overlap_all) > 0:
        print("\nMacierz korelacji:")
        corr_matrix = overlap_all[baseline_cols].corr()
        print(corr_matrix.to_string())
        
        print("\nŚrednie wartości:")
        for col in baseline_cols:
            print(f"  {col[:40]}...: {overlap_all[col].mean():.3f}")
else:
    print("Nie wszystkie kolumny baseline dostępne w dataframe")


## Prolactin – Column Consolidation

| Priority | Column | Group | N | Action |
|----------|--------|-------|---|--------|
| 1 | `PRL godz. 10:00 (PRL10)` | PCOS | 1165 | Primary |
| 2 | `PRL ng/ml` | Control | 45 | Fill missing |

→ Merge → `prolactin_ng_ml` (N = 1210)

**Note:** Significant group differences exist (PCOS median 10.2 vs Control 16.7, p < 10⁻¹¹). Use with caution in pooled analyses; consider including group as a covariate.

In [ ]:
print("=" * 80)
print("PRL COLUMNS – DESCRIPTIVE STATISTICS")
print("=" * 80)

# List of  columns
prl_columns = [
    'PRL godz. 10:00 (PRL10)',
    'PRL ng/ml',
]

# Create description dataframe
prl_desc = []

for col in prl_columns:
    if col in df.columns:
        series = df[col].dropna()
        prl_desc.append({
            'Column': col,
            'N': len(series),
            'Mean': series.mean(),
            'Std': series.std(),
            'Min': series.min(),
            '25%': series.quantile(0.25),
            'Median': series.median(),
            '75%': series.quantile(0.75),
            'Max': series.max()
        })

prl_desc_df = pd.DataFrame(prl_desc)

# Display
print(prl_desc_df.to_string(index=False))

# Save to CSV
prl_desc_df.to_csv('prl_columns_description.csv', index=False)
print("\n✅ Saved to 'prl_columns_description.csv'")

In [ ]:
from scipy.stats import mannwhitneyu

pcos = df['PRL godz. 10:00 (PRL10)'].dropna()
ctrl = df['PRL ng/ml'].dropna()

mannwhitneyu(pcos, ctrl, alternative='two-sided')

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot([pcos, ctrl], labels=['PCOS', 'Control'])
plt.ylabel('PRL ng/mL')
plt.show()

In [ ]:
# Sprawdź overlap między dwiema kolumnamie rbc w PCOS
overlap_rbc = df[df['Morfologia CBC (RBC)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RBC)'].notna()]
print(f"Pacjentki z oboma pomiarami rbc: {len(overlap_rbc)}")

if len(overlap_rbc) > 0:
    # Sprawdź korelację
    corr = overlap_rbc['Morfologia CBC (RBC)'].corr(
           overlap_rbc['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RBC)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")

## RDW – Column Consolidation

### Descriptive Statistics

| Column | N | Median | Unit |
|--------|---|--------|------|
| `Morfologia CBC (RDW)` | 44 | 12.55 | % |
| `Morfologia krwi... (RDW)` | 901 | 12.60 | % |
| `RDW %` | 45 | 12.60 | % |
| `Morfologia CBC (RDWSD)` | 44 | 40.25 | fL |
| `Morfologia krwi... (RDWSD)` | 901 | 41.60 | fL |

### Overlap Check

| Comparison | Overlap |
|------------|---------|
| RDW (%) columns (CBC vs diff vs Control) | **0 patients** |
| RDWSD (fL) columns (CBC vs diff) | **0 patients** |

### Decision

| Parameter | Columns | Unit | Action |
|-----------|---------|------|--------|
| **RDW** | `Morfologia CBC (RDW)` + `Morfologia krwi... (RDW)` + `RDW %` | % | Merge → `rdw_percent` |
| **RDWSD** | `Morfologia CBC (RDWSD)` + `Morfologia krwi... (RDWSD)` | fL | Merge → `rdwsd_fL` |

> No overlap detected – columns come from different patient subgroups. Merge is safe.

In [ ]:
print("=" * 80)
print("RDW COLUMNS – DESCRIPTIVE STATISTICS")
print("=" * 80)

# List of RDW columns
rdw_columns = [
    'Morfologia CBC (RDW)',
    'Morfologia CBC (RDWSD)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDW)',
    'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDWSD)',
    'RDW %'
]

# Create description dataframe
rdw_desc = []

for col in rdw_columns:
    if col in df.columns:
        series = df[col].dropna()
        rdw_desc.append({
            'Column': col,
            'N': len(series),
            'Mean': series.mean(),
            'Std': series.std(),
            'Min': series.min(),
            '25%': series.quantile(0.25),
            'Median': series.median(),
            '75%': series.quantile(0.75),
            'Max': series.max(),
            'Unit': 'fL' if 'RDWSD' in col else '%'
        })

rdw_desc_df = pd.DataFrame(rdw_desc)

# Display
print(rdw_desc_df.to_string(index=False))

# Save to CSV
rdw_desc_df.to_csv('rdw_columns_description.csv', index=False)
print("\n✅ Saved to 'rdw_columns_description.csv'")

# ============================================================================
# OVERLAP CHECK
# ============================================================================
print("\n" + "=" * 80)
print("RDW COLUMNS – OVERLAP CHECK")
print("=" * 80)

# Separate by unit type
rdw_percent_cols = [c for c in rdw_columns if c in df.columns and 'RDWSD' not in c]
rdw_fl_cols = [c for c in rdw_columns if c in df.columns and 'RDWSD' in c]

# Check overlap within percent columns
print("\n1. RDW (%) columns overlap:")
for i, col1 in enumerate(rdw_percent_cols):
    for col2 in rdw_percent_cols[i+1:]:
        overlap = df[df[col1].notna() & df[col2].notna()]
        print(f"   {col1[:40]}... vs {col2[:40]}...: {len(overlap)} pacjentek")

# Check overlap within fL columns
print("\n2. RDW (fL) columns overlap:")
for i, col1 in enumerate(rdw_fl_cols):
    for col2 in rdw_fl_cols[i+1:]:
        overlap = df[df[col1].notna() & df[col2].notna()]
        if len(overlap) > 0:
            corr = overlap[col1].corr(overlap[col2])
            print(f"   {col1[:40]}... vs {col2[:40]}...: {len(overlap)} pacjentek, korr={corr:.3f}")
        else:
            print(f"   {col1[:40]}... vs {col2[:40]}...: {len(overlap)} pacjentek")


In [ ]:
# Sprawdź overlap między dwiema kolumnamie wbc w PCOS
overlap_wbc = df[df['Morfologia CBC (WBC)'].notna() & df['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (WBC)'].notna()]
print(f"Pacjentki z oboma pomiarami wbc: {len(overlap_wbc)}")

if len(overlap_wbc) > 0:
    # Sprawdź korelację
    corr = overlap_wbc['Morfologia CBC (WBC)'].corr(
           overlap_wbc['Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (WBC)'])
    print(f"Korelacja między pomiarami: {corr:.3f}")
                                                                                   

In [ ]:
# TODO: Poprawić ręczne przypisanie kategorii do Testosteron Total

In [ ]:
print("=" * 80)
print("COLUMN CONSOLIDATION – MERGING PARAMETERS & UPDATING QUALITY DICTIONARY")
print("=" * 80)

# Create a new DataFrame for merged parameters (start with original df)
df_merged = df.copy()

# Define mapping: new_variable -> list of source columns
merge_mapping = {
    'AMH_ng_ml': [
        'AMH (hormon anty-Mullerowski) (AMH_CP)',
        'AMH-anty Mullerian Hormon (AMH)',
        'AMH ng/ml'
    ],
    'androstenedione_ng_ml': [
        'Androstendion (ANDRO)',
        'Androstendion (I31)',
        'Androstendion (I31) (ANDRO)'
    ],
    'anti_tg_IU_mL': [
        'P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)',
        'Anty-TG (p/c przeciw tyreoglobulinie) (ATG)'
    ],
    'cortisol_8am_ug_dL': [
        'Kortyzol godz. 08:00 (KOR)',
        'kortyzol 8.00 ug/dl'
    ],
    'dheas_ug_dL': [
        'DHEAS (DHEA)',
        'DHEAS ug/dl'
    ],
    'fsh_iu_l': [
        'FSH IU/l',
        'FSH (FSH)'
    ],
    'glucose_fasting_mg_dL': [
        'Krzywa cukrowa - 2 punktowa (L_GLU_0)',
        "glu 0' mg/dl"
    ],
    'hematocrit_percent': [
        'Morfologia CBC (HCT)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HCT)'
    ],
    'hemoglobin_g_dL': [
        'Morfologia CBC (HGB)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (HGB)'
    ],
    'insulin_fasting_uU_ml': [
        'Insulina (INSUL)',
        'Insulina po 75g glukozy (3pkt.) (INSUL_0)',
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 0m)',
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)',
        'insulina uU/ml'
    ],
    'insulin_30min_uU_ml': [
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 30m)',
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_1)'
    ],
    'insulin_60min_3pt_uU_ml': [
        'Insulina po 75g glukozy (3pkt.) (INSUL_1)'
    ],
    'insulin_60min_4pt_uU_ml': [
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 60m)',
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_2)'
    ],
    'insulin_120min_3pt_uU_ml': [
        'Insulina po 75g glukozy (3pkt.) (INSUL_2)'
    ],
    'insulin_120min_4pt_uU_ml': [
        'Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97 120m)'
    ],
    'lh_iu_l': [
        'LH (LH)',
        'LH IU/l'
    ],
    'lymphocytes_abs_10_3_uL': [
        'Limf 10^3/uL',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (LIM#)'
    ],
    'mch_pg': [
        'Morfologia CBC (MCH)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCH)'
    ],
    'mchc_g_dL': [
        'Morfologia CBC (MCHC)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCHC)'
    ],
    'mcv_fL': [
        'Morfologia CBC (MCV)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MCV)'
    ],
    'mpv_fL': [
        'MPV fL',
        'Morfologia CBC (MPV)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MPV)'
    ],
    'pct_percent': [
        'Morfologia CBC (PCT)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PCT)'
    ],
    'pdw_fL': [
        'Morfologia CBC (PDW)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PDW)'
    ],
    'plcr_percent': [
        'Morfologia CBC (PLCR)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLCR)'
    ],
    'neutrophils_abs_10_3_uL': [
        'Neu 10^3/uL',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT#)'
    ],
    'platelets_10_3_uL': [
        'PLT 10^3/uL',
        'Morfologia CBC (PLT)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLT)'
    ],
    '17ohp_baseline_ng_ml': [
        '17 - OH progesteron (L79) (17-OHPG)',
        '17 OH progesteron (L79)'
    ],
    'prolactin_ng_ml': [
        'PRL godz. 10:00 (PRL10)',
        'PRL ng/ml'
    ],
    'rbc_10_6_uL': [
        'Morfologia CBC (RBC)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RBC)'
    ],
    'rdw_percent': [
        'Morfologia CBC (RDW)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDW)',
        'RDW %'
    ],
    'rdwsd_fL': [
        'Morfologia CBC (RDWSD)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDWSD)'
    ],
    'shbg_nmol_l': [
        'SHBG nmol/l',
        'SHBG (L_SHGB)'
    ],
    'testosterone_ng_ml': [
        'testosteron ng/ml',
        'Testosteron (L_TESTOS)'
    ],
    'testosterone_free_nmol_L': [
        'Testosteron wolny (O41) (TEST-F)',
        'Wolny testosteron (O41.W)'
    ],
    'tsh_uIU_mL': [
        'TSH (TSH)',
        'TSH uIU/ml'
    ],
    'wbc_10_3_uL': [
        'Morfologia CBC (WBC)',
        'Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (WBC)',
        'WBC 10^3/uL'
    ]
}

# Perform merging and update parameter_quality
merged_count = 0
all_source_cols = set()

for new_var, source_cols in merge_mapping.items():
    # Filter existing columns
    existing_sources = [col for col in source_cols if col in df_merged.columns]
    if not existing_sources:
        print(f"⚠️ No source columns found for {new_var}")
        continue
    
    # Create merged column
    df_merged[new_var] = df_merged[existing_sources[0]]
    for col in existing_sources[1:]:
        df_merged[new_var] = df_merged[new_var].fillna(df_merged[col])
    
    # Mark source columns as obsolete in parameter_quality
    for col in existing_sources:
        if col in parameter_quality:
            parameter_quality[col]['useful'] = False
            parameter_quality[col]['exclude_reason'].append(f'merged into {new_var}')
            parameter_quality[col]['recommended_action'] = 'merged'
            parameter_quality[col]['merged_into'] = new_var
            all_source_cols.add(col)
            
    # Determine only_in_group for the new variable
    source_groups = set()
    for col in existing_sources:
        group = parameter_quality.get(col, {}).get('only_in_group')
        if group:
            source_groups.add(group)

    if len(source_groups) == 1:
        only_in_group = source_groups.pop()
    elif len(source_groups) > 1:
        only_in_group = None  # mixed PCOS + Control
    else:
        only_in_group = None  # unknown
    
    # Mark new variable as useful
    parameter_quality[new_var] = {
        'original_name': new_var,
        'data_type': 'float64',
        'n_total': len(df_merged),
        'n_nonnull': df_merged[new_var].notna().sum(),
        'n_unique': df_merged[new_var].nunique(),
        'useful': True,
        'exclude_reason': [],
        'only_in_group': only_in_group,
        'too_few_data': False,
        'no_variance': False,
        'formatting_errors': False,
        'unit_unknown': False,
        'outliers_detected': False,
        'detected_unit': parameter_quality.get(existing_sources[0], {}).get('detected_unit', None),        'category': parameter_quality.get(existing_sources[0], {}).get('category', 'Unknown'),
        'recommended_action': 'keep',
        'notes': [f'Consolidated from: {", ".join(existing_sources)}'],
        'merged_from': existing_sources
    }
    
    merged_count += 1
    print(f"✓ {new_var} (from {len(existing_sources)} columns)")

print(f"\n✅ Created {merged_count} consolidated variables")
print(f"✅ Marked {len(all_source_cols)} source columns as obsolete")

# ============================================================================
# VERIFICATION
# ============================================================================
print("\n" + "=" * 80)
print("VERIFICATION – USEFUL PARAMETERS AFTER CONSOLIDATION")
print("=" * 80)

useful_after = [col for col, info in parameter_quality.items() if info.get('useful') == True]
print(f"\n✅ Useful parameters: {len(useful_after)}")

# Display first 30 useful parameters
print(f"\nAll useful parameters [{len(useful_after)}]:")
for col in useful_after:
    info = parameter_quality[col]
    print(f"  • {col} [{info.get('category', 'Unknown')}] – {info.get('n_nonnull', 0)} values")

# Save updated DataFrame
df_merged.to_csv('df_merged.csv', index=False)
print("\n✅ Saved merged DataFrame to 'df_merged.csv'")

In [ ]:
print("=" * 80)
print("FULL PARAMETER QUALITY DICTIONARY – ALL KEYS")
print("=" * 80)

# Collect all possible keys from all parameter dictionaries
all_keys = set()
for info in parameter_quality.values():
    all_keys.update(info.keys())

# Create DataFrame from all parameters
all_params_df = pd.DataFrame([
    {**{'Parameter': col}, **info}
    for col, info in parameter_quality.items()
])

# Reorder columns to put Parameter first
cols = ['Parameter'] + sorted([k for k in all_keys if k != 'Parameter'])
all_params_df = all_params_df[[c for c in cols if c in all_params_df.columns]]

# Display
print(f"\n📊 Total parameters: {len(all_params_df)}\n")
print(all_params_df.to_string(index=False))

# Save to CSV
all_params_df.to_csv('full_parameter_quality_dictionary.csv', index=False)
print("\n✅ Saved to 'full_parameter_quality_dictionary.csv'")

# Also display a compact view of useful vs excluded
print("\n" + "=" * 80)
print("COMPACT SUMMARY – USEFUL VS EXCLUDED")
print("=" * 80)

useful_count = sum(1 for info in parameter_quality.values() if info.get('useful') == True)
excluded_count = sum(1 for info in parameter_quality.values() if info.get('useful') == False)
total_count = len(parameter_quality)

print(f"\n  ✅ Useful: {useful_count}")
print(f"  ❌ Excluded: {excluded_count}")
print(f"  📊 Total: {total_count}")

# Show distribution of exclude reasons
print("\n" + "=" * 80)
print("EXCLUDE REASONS DISTRIBUTION")
print("=" * 80)

exclude_reasons = {}
for info in parameter_quality.values():
    for reason in info.get('exclude_reason', []):
        # Simplify reason for counting
        if 'too few data' in reason:
            simple_reason = 'too few data (N < 30)'
        elif 'only in' in reason:
            simple_reason = reason
        elif 'no variance' in reason:
            simple_reason = 'no variance (≤2 unique values)'
        elif 'merged into' in reason:
            simple_reason = 'merged into consolidated variable'
        elif 'formatting' in reason.lower():
            simple_reason = 'formatting errors'
        else:
            simple_reason = reason
        exclude_reasons[simple_reason] = exclude_reasons.get(simple_reason, 0) + 1

for reason, count in sorted(exclude_reasons.items(), key=lambda x: -x[1]):
    print(f"  {reason}: {count}")

<span style="color:gray">

## BARDZO DRAFT!!! [po zakresy wartosci akceptowalnych trzeba luknac po jej pracach z ctrl+f i hasłem "outlier"]: Outlier Detection and Biological Plausibility Filtering

### Overview

Outliers were handled using two distinct approaches:

1. **Biological plausibility filtering** – values outside clinically meaningful ranges were replaced with `NaN`
2. **Statistical outlier detection (IQR)** – outliers were flagged and counted but **not removed** from the dataset

This dual approach ensures that biologically impossible values are excluded while preserving true biological variation (including PCOS-related extremes).

---

### 1. Biological Plausibility Filtering

For selected clinical parameters, biologically plausible ranges were defined based on established reference ranges for women of reproductive age, with adjustments for PCOS-specific elevations.

#### Defined Ranges (`bio_limits`)

| Parameter | Lower Bound | Upper Bound | Rationale |
|-----------|-------------|-------------|-----------|
| Testosterone | 0 | 10 ng/mL | PCOS may elevate testosterone, but >10 ng/mL suggests other pathology |
| DHEAS | 0 | 1000 μg/dL | Adrenal androgens may be elevated in PCOS |
| Androstenedione | 0 | 20 ng/mL | PCOS may elevate androstenedione |
| SHBG | 1 | 300 nmol/L | Very low SHBG (<1) or very high (>300) are biologically implausible |
| FSH | 0.1 | 50 IU/L | Ovarian failure typically >40; PCOS usually normal/low |
| LH | 0.1 | 80 IU/L | PCOS may have elevated LH but rarely >80 |
| AMH | 0 | 30 ng/mL | PCOS often has elevated AMH |
| Prolactin | 0 | 200 ng/mL | Hyperprolactinemia defined as >25-30; >200 suggests macroprolactinoma |
| Platelets (PLT) | 50 | 1000 × 10³/uL | Severe thrombocytopenia or thrombocytosis |
| MPV | 5 | 20 fL | Mean platelet volume beyond this range is unlikely |
| RDW | 8 | 25 % | Red cell distribution width beyond this range suggests other pathology |

#### Action

For each parameter with defined bounds:

```python
mask = (df[param] < lower) | (df[param] > upper)
df.loc[mask, param] = np.nan
```

- Values outside the defined range were replaced with `NaN`
- No whole-patient exclusion occurred at this stage
- Each replacement was logged with parameter name, number of affected values, and bounds used

#### Example

For testosterone with a measured value of 15 ng/mL (exceeding the upper bound of 10 ng/mL):

```python
# Before: df['testosterone'] = 15.0
mask = df['testosterone'] > 10
df.loc[mask, 'testosterone'] = np.nan
# After: df['testosterone'] = NaN
```

#### Output

- `biological_plausibility_log.csv` – log of all biologically implausible values replaced, containing:
  - Parameter name
  - Number of values replaced
  - Lower and upper bounds applied
  - Percentage of total observations affected

---

### 2. Statistical Outlier Detection (IQR)

Outliers were identified using the **interquartile range (IQR)** method but were **not removed** from the dataset. This approach preserves true biological variation while documenting extreme values for sensitivity analyses.

#### Method

For each numeric parameter:

- **Lower fence:** `Q1 - 1.5 × IQR`
- **Upper fence:** `Q3 + 1.5 × IQR`

```python
Q1 = df[param].quantile(0.25)
Q3 = df[param].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df[param] < lower_fence) | (df[param] > upper_fence)]
```

#### Action

- Outliers were **flagged and counted** only (`action = 'flag_only'`)
- **No values were replaced or winsorized**
- Outliers were documented for transparency and sensitivity analysis

#### Rationale for Not Removing Statistical Outliers

| Reason | Explanation |
|--------|-------------|
| **PCOS heterogeneity** | True biological variation (e.g., severe insulin resistance, very high LH, extreme hyperandrogenism) defines specific phenotypes and should be preserved |
| **Clinical extremes matter** | Very high testosterone or AMH may identify distinct PCOS subtypes that are clinically relevant |
| **Sample size preservation** | Removing outliers reduces statistical power, particularly problematic in subgroup analyses or with rare phenotypes |
| **Downstream method compatibility** | Clustering algorithms (e.g., k-means, hierarchical) and regression models can be run with and without outliers to assess robustness |
| **Transparency** | Flagging without removal allows readers to assess the impact of outliers on results |

#### Output

- `statistical_outliers_IQR_log.csv` – log of all statistical outliers by parameter, containing:
  - Parameter name
  - Number of outliers detected (lower and upper separately)
  - Q1, Q3, IQR, and fence values
  - Percentage of observations flagged as outliers

---

### 3. What Actually Gets Removed from Analysis

| Stage | Action | Impact |
|-------|--------|--------|
| **Biological plausibility** | Values outside `bio_limits` → `NaN` | ❌ Specific values only (converted to missing) |
| **Column-level missingness** | Columns with >30% missing values (except protected variables) | ❌ Entire columns excluded from analysis |
| **Statistical outliers (IQR)** | Flagged only | ❌ No removal |
| **Complete-case analysis (sensitivity)** | `df_cc = df.dropna(subset=required_vars)` | ⚠️ Whole patients excluded only for secondary sensitivity analysis |

#### Protected Variables

Certain key clinical variables were **excluded from column-level exclusion** even if missing >30%:
- `LIM#` (lymphocyte absolute count)
- `NEUT#` (neutrophil absolute count)
- `PLT` (platelet count)
- `MPV` (mean platelet volume)

These were preserved due to their clinical importance in PCOS phenotyping.

---

### 4. Summary Table

| Outlier Type | Detection Method | Action | Values Removed? | Patients Removed? | Logged? |
|--------------|------------------|--------|-----------------|-------------------|---------|
| Biologically implausible | Fixed clinical ranges | Replace with `NaN` | ✅ Yes (specific values only) | ❌ No | ✅ |
| Statistical (IQR) | `Q1 - 1.5×IQR` / `Q3 + 1.5×IQR` | Flag only | ❌ No | ❌ No | ✅ |
| Column-level missing | >30% missing (non-protected) | Drop column | ❌ No (whole column) | ❌ No | ✅ |

---

### 5. Sensitivity Analysis Plan

To assess the impact of outlier handling decisions, the following sensitivity analyses will be conducted:

1. **Primary analysis:** Biologically implausible values removed (`NaN`), statistical outliers retained
2. **Sensitivity analysis A:** Both biologically implausible values AND statistical outliers removed
3. **Sensitivity analysis B:** Neither removed (original data)
4. **Sensitivity analysis C:** Winsorization at 1st and 99th percentiles

Results will be compared across all four approaches. If results are consistent, the primary analysis (most conservative) will be reported.

---

### 6. Next Steps After Outlier Handling

1. ✅ Review biological plausibility logs – confirm replaced values are indeed implausible
2. ⬜ Run sensitivity analyses – compare results with vs. without statistical outliers
3. ⬜ Proceed to missing data imputation (mean/median, KNN, or MICE)
4. ⬜ Normalization before clustering (Z-score, Min-Max, or log transformation for skewed distributions)

```

